# global spectral geometry + dominant-mode spatial distribution FUSION V2 — EARLY & LATE
### Optimizado para maximizar balanced_accuracy y F1-macro

---

## ¿Qué mejoras tiene esta versión respecto a V1?

| Área | V1 | V2 |
|---|---|---|
| **RFF gammas** | [0.25, 0.5, 1.0] | [0.10, 0.25, 0.50, 1.0, 2.0, 4.0] |
| **RFF componentes** | 400 | 600 |
| **RFF + clasificador** | solo LR | LR + LinearSVC + SGD |
| **SVM RBF** | C=1 fijo | grid C=[0.1, 1, 5, 10, 50] |
| **Tree ensemble** | ExtraTrees 1200 | ExtraTrees + **HistGradientBoosting** |
| **Selección features** | ninguna | **SelectKBest** (f_classif, k=[6,10,16]) antes de RFF |
| **Threshold tuning** | f1_macro | f1_macro + **subj_f1** |
| **Meta-stacking** | LR 2 features | LR + Ridge + SGD, **N features = N modelos base** |
| **Stacking features** | p_h1, p_h2 | p_h1, p_h2 + scores individuales de cada modelo |
| **Early fusion sets** | 5 | 8 (añade variantes PVT-only y delta+static) |
| **Calibración** | solo LinearSVC | todos los modelos que usan `decision_function` |

## Estructura
| Sección | Contenido |
|---|---|
| 1 | CONFIG global y rutas |
| 2 | Feature sets global spectral geometry y dominant-mode spatial distribution |
| 3 | Helpers compartidos (CV, métricas, limpieza) |
| 4 | Modelos — catálogo completo |
| 5 | Carga y preparación de datos |
| 6 | **EARLY FUSION** |
| 7 | **LATE FUSION** |
| 8 | Main unificado |
| **9**  | **Calibración isotónica + Rank Averaging** | calibra probs antes del blend |
| **10** | **Selección dinámica de experto (DCS-kNN + Router)** | elige global spectral geometry o dominant-mode spatial distribution por sujeto |
| **11** | **Early Fusion con features de interacción global spectral geometry×dominant-mode spatial distribution** | productos cruzados interaction terms between global spectral geometry and dominant-mode spatial distribution |
| **12** | **Main extendido → FUSION_ML_COMPLETO** | ejecuta todo |


This notebook is part of the public analysis repository associated with the EEG eigenmode study. It uses precomputed feature tables generated by the preprocessing and feature extraction scripts. Raw EEG recordings and participant-level data are not included because the study involves minors and is subject to ethical and privacy restrictions.

---
## SECCIÓN 1 — Imports y CONFIG

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from pathlib import Path
import warnings
from collections import Counter
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGK = True
except Exception:
    HAS_SGK = False

warnings.filterwarnings("ignore")

# ── RUTAS ──────────────────────────────────────────────────────────────────────
BASE = Path(r"")

GSG_DIR = BASE / "results/features/global_spectral_geometry"
DMSD_DIR = BASE / "results/features/dominant_mode_spatial_distribution"

GSG_BYWIN       = GSG_DIR / "01_features" / "gsg_features_by_win.csv"
GSG_POOLED      = GSG_DIR / "01_features" / "gsg_features_pooled.csv"
GSG_DELTA_BYWIN = GSG_DIR / "07_delta_basal_to_pvt" / "gsg_delta_table_by_win.csv"

DMSD_BYWIN       = DMSD_DIR / "01_features" / "dmsd_features_por_ventana.csv"
DMSD_POOLED      = DMSD_DIR / "01_features" / "dmsd_features_pooled.csv"
DMSD_DELTA_BYWIN = DMSD_DIR / "01_features" / "dmsd_delta_por_ventana.csv"

# Nombres alternativos (por si los CSVs se llamaron de otra forma)
if not DMSD_BYWIN.exists():   DMSD_BYWIN   = DMSD_DIR / "01_features" / "dmsd_features_by_win.csv"
if not DMSD_DELTA_BYWIN.exists(): DMSD_DELTA_BYWIN = DMSD_DIR / "01_features" / "dmsd_delta_by_win.csv"

OUT_DIR_EARLY = BASE / "EIGENMODE_combined eigenmode representations_EARLY_FUSION_V2"
OUT_DIR_LATE  = BASE / "EIGENMODE_combined eigenmode representations_LATE_FUSION_V3"
OUT_DIR_EARLY.mkdir(parents=True, exist_ok=True)
OUT_DIR_LATE.mkdir(parents=True, exist_ok=True)

# ── HIPERPARÁMETROS GLOBALES ───────────────────────────────────────────────────
CSV_FLOAT_FMT  = "%.8e"
SEED           = 123

N_SPLITS_DESIRED       = 5
MIN_ROWS_PER_BLOCK     = 20
MIN_SUBJECTS_PER_BLOCK = 12
MIN_GROUPS_PER_CLASS   = 2

MAX_NAN_FRAC_PER_FEATURE = 0.40
MIN_VAR_PER_FEATURE      = 1e-12
MIN_FEATURES_AFTER_CLEAN = 4

# RFF: más gammas y más componentes que en V1
# Gammas pequeñas → kernel suave (bueno si features están bien escaladas)
# Gammas grandes  → kernel local (bueno si hay clusters muy distintos)
RFF_GAMMAS     = [0.10, 0.25, 0.50, 1.0, 2.0, 4.0]
RFF_COMPONENTS = 600   # +50% respecto a V1 → mejor aproximación del kernel exacto

# SelectKBest: probar estos valores de k antes del RFF
# Reduce dimensionalidad antes de expandir con RFF → menos ruido, más señal
SKB_K_VALUES   = [6, 10, 16, "all"]  # "all" = sin selección

# SVM C grid: busca el mejor C dentro de cada fold
SVM_C_GRID     = [0.1, 1.0, 5.0, 10.0, 50.0]

THRESH_GRID    = np.linspace(0.25, 0.75, 21)  # más pasos que V1
BLEND_WEIGHTS_GSG = [0.15, 0.25, 0.35, 0.45, 0.50, 0.55, 0.65, 0.75, 0.85]
DMSD_PIPELINES   = ["all_channels", "no_occipital"]
MIN_VALID_FOLDS_FOR_MAIN_RANKING = 4

print("CONFIG OK")
print(f"  OUT early  → {OUT_DIR_EARLY}")
print(f"  OUT late   → {OUT_DIR_LATE}")
print(f"  Modelos RFF: {len(RFF_GAMMAS)} gammas × 3 clasificadores = {len(RFF_GAMMAS)*3} modelos RFF")
print(f"  SKB k-values: {SKB_K_VALUES}")

---
## SECCIÓN 2 — Feature sets global spectral geometry y dominant-mode spatial distribution

In [ ]:
# ── global spectral geometry ─────────────────────────────────────────────────────────────────────────
GSG_PRIORITY = [
    "all__dist_prop_gt_0.02", "all__dist_p95", "osc_frac",
    "osc__rad_signed_median", "osc__rad_prop_out", "osc__rad_prop_in",
    "osc__dist_uc_abs_mean", "osc__rad_signed_mean", "osc__imag_abs_mean",
    "osc__ang_abs_std_deg", "osc__ring0.02__angle_prop_gt_35deg",
    "osc__ring0.05__angle_prop_gt_35deg", "osc__angle_p95_deg",
    "osc__angle_prop_gt_35deg", "osc__dist_prop_gt_0.05",
    "osc__angle_prop_gt_aacc_p95", "all__dist_prop_gt_aacc_p95",
]

GSG_STAT_COMPACT = [
    "all__dist_prop_gt_0.02", "all__dist_p95", "osc_frac",
    "osc__rad_signed_median", "osc__dist_uc_abs_mean", "osc__imag_abs_mean",
    "osc__ang_abs_std_deg", "osc__angle_p95_deg", "osc__rad_p95",
    "osc__rad_std", "all__rad_prop_out", "osc__dist_uc_abs_std",
]

# NUEVO: versión mínima con solo las features más discriminativas
# Útil cuando el espacio de features es pequeño y el ruido daña
GSG_CORE_ONLY = [
    "all__dist_prop_gt_0.02", "osc_frac",
    "osc__rad_signed_median", "osc__imag_abs_mean",
    "osc__ang_abs_std_deg", "all__dist_p95",
]

GSG_DELTA_FOCUS = [
    "delta__all__rad_signed_median", "delta__all__rad_prop_out",
    "delta__all__dist_uc_abs_mean", "delta__osc_frac",
    "delta__osc__imag_abs_mean", "delta__osc__ring0.02__angle_prop_gt_35deg",
    "delta__osc__ring0.05__angle_prop_gt_35deg", "delta__all__dist_prop_gt_0.02",
    "delta__all__dist_p95", "delta__osc__rad_prop_out",
    "delta__osc__rad_signed_median", "delta__osc__dist_uc_abs_mean",
]

# ── dominant-mode spatial distribution ─────────────────────────────────────────────────────────────────────────
DMSD_DISTRIBUTION = [
    "max_p", "top4_mass", "gini", "kurtosis_p",
    "core_to_rest", "dom_max_region", "dom_gap_core_vs_best_other",
]

DMSD_PRIORITY_STAT = [
    "s_core", "s_core_star", "logit_core", "sD_core",
    "core_to_rest", "dom_gap_core_vs_best_other",
    "s_temporal", "logit_temporal",
]

DMSD_PRIORITY_NO_OCCIPITAL_VISUAL = [
    "s_core", "s_core_star", "s_temporal", "logit_core", "logit_temporal",
    "sD_core", "sD_rest", "core_to_rest", "dom_gap_core_vs_best_other",
]

# NUEVO: dominant-mode spatial distribution ampliado con métricas de concentración global
DMSD_FULL = [
    "s_core", "s_core_star", "logit_core", "sD_core",
    "core_to_rest", "dom_gap_core_vs_best_other",
    "s_temporal", "logit_temporal", "s_frontal", "s_parietal", "s_central",
    "entropy_norm", "hhi", "n_eff", "gini",
    "max_p", "top4_mass", "kurtosis_p",
]

DMSD_PRIORITY_DELTA = [
    "delta_s_core", "delta_sD_core", "delta_logit_core",
    "delta_core_to_rest", "delta_s_core_star", "delta_rest_shift",
    "delta_s_temporal", "delta_s_temporal_star",
]

DMSD_DELTA_REORG = [
    "delta_reorg_L1", "delta_reorg_L2",
    "delta_core_vs_central_shift", "delta_rest_shift",
]

# NUEVO: delta ampliado
DMSD_DELTA_FULL = DMSD_PRIORITY_DELTA + DMSD_DELTA_REORG + [
    "delta_entropy_norm", "delta_hhi", "delta_gini",
]

print("Feature sets OK")
print(f"  GSG_PRIORITY: {len(GSG_PRIORITY)} features")
print(f"  DMSD_FULL:     {len(DMSD_FULL)} features")

---
## SECCIÓN 3 — Helpers compartidos

In [ ]:
# ══════════════════════════════════════════════════════════════
# UTILIDADES GENERALES
# ══════════════════════════════════════════════════════════════

def dedupe_keep_order(seq):
    seen, out = set(), []
    for x in seq:
        if x not in seen: out.append(x); seen.add(x)
    return out

def col_as_series(df, colname):
    x = df.loc[:, colname]
    return x.iloc[:, 0] if isinstance(x, pd.DataFrame) else x

def available_features(df, candidates):
    return dedupe_keep_order([c for c in candidates if c in df.columns])

def build_X_numeric(df_block, feature_cols):
    X = pd.DataFrame(index=df_block.index)
    for c in feature_cols:
        X[c] = pd.to_numeric(col_as_series(df_block, c), errors="coerce")
    return X

def most_frequent_nonempty(series):
    vals = [str(x) for x in series if pd.notna(x) and str(x).strip() != ""]
    return Counter(vals).most_common(1)[0][0] if vals else ""

# ══════════════════════════════════════════════════════════════
# VALIDACIÓN CRUZADA
# ══════════════════════════════════════════════════════════════

def make_group_cv(y, groups, n_splits_desired=5):
    y      = np.asarray(y).astype(int)
    groups = np.asarray(groups).astype(str)
    n_splits = int(min(max(2, n_splits_desired), len(np.unique(groups))))
    if n_splits < 2: return None
    if HAS_SGK:
        tmp = pd.DataFrame({"g": groups, "y": y}).drop_duplicates("g")
        if int((tmp["y"]==0).sum()) >= n_splits and int((tmp["y"]==1).sum()) >= n_splits:
            return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    return GroupKFold(n_splits=n_splits)

# ══════════════════════════════════════════════════════════════
# MÉTRICAS
# ══════════════════════════════════════════════════════════════

def safe_auc(y_true, y_score):
    try: return roc_auc_score(y_true, y_score)
    except: return np.nan

def to_prob_like(score):
    score = np.asarray(score, float)
    if score.size == 0: return score
    if np.nanmin(score) < 0 or np.nanmax(score) > 1:
        score = 1.0 / (1.0 + np.exp(-score))
    return score

def choose_best_threshold(y_true, y_score, grid=THRESH_GRID):
    """
    Busca el umbral óptimo maximizando f1_macro.
    También prueba maximizar balanced_accuracy por separado
    y devuelve el mejor de los dos.
    """
    best_thr, best_f1 = 0.5, -np.inf
    y_score = np.asarray(y_score, float)
    for thr in grid:
        yhat = (y_score >= thr).astype(int)
        try:
            f1m = f1_score(y_true, yhat, average="macro")
            bal = balanced_accuracy_score(y_true, yhat)
            # combinar f1_macro y bal_acc para elegir umbral
            score_combined = 0.6 * f1m + 0.4 * bal
        except: continue
        if np.isfinite(score_combined) and score_combined > best_f1:
            best_f1, best_thr = score_combined, float(thr)
    return best_thr, best_f1

def aggregate_subject_scores(scores, ids):
    return (pd.DataFrame({"id": ids, "score": scores})
              .groupby("id", as_index=False)["score"].mean())

def evaluate_scores(y_true, ids, score, thr):
    """Métricas a nivel ventana Y a nivel sujeto (lo más importante)."""
    score = np.asarray(score, float)
    yhat  = (score >= thr).astype(int)

    row_bal = balanced_accuracy_score(y_true, yhat)
    row_f1  = f1_score(y_true, yhat, average="macro")
    row_auc = safe_auc(y_true, score)

    agg   = aggregate_subject_scores(score, ids)
    y_sub = pd.DataFrame({"id": ids, "y": y_true}).groupby("id", as_index=False)["y"].first()
    m     = agg.merge(y_sub, on="id", how="inner")

    subj_bal = subj_f1 = subj_auc = subj_n = np.nan
    if len(m) >= 4:
        m["yhat"] = (m["score"].values >= thr).astype(int)
        subj_bal = balanced_accuracy_score(m["y"].values, m["yhat"].values)
        subj_f1  = f1_score(m["y"].values, m["yhat"].values, average="macro")
        subj_auc = safe_auc(m["y"].values, m["score"].values)
        subj_n   = len(m)

    return dict(bal_acc=row_bal, f1_macro=row_f1, auc=row_auc,
                subj_bal_acc=subj_bal, subj_f1_macro=subj_f1,
                subj_auc=subj_auc, subj_n_test=subj_n)

# ══════════════════════════════════════════════════════════════
# LIMPIEZA DE FEATURES
# ══════════════════════════════════════════════════════════════

def clean_feature_block(df_block, feature_cols):
    keep, dropped = [], []
    feature_cols = dedupe_keep_order([c for c in feature_cols if c in df_block.columns])
    for c in feature_cols:
        x  = pd.to_numeric(col_as_series(df_block, c), errors="coerce").values.astype(float)
        xf = x[np.isfinite(x)]
        nan_frac = float(np.mean(~np.isfinite(x)))
        var = float(np.var(xf)) if xf.size > 0 else np.nan
        if nan_frac > MAX_NAN_FRAC_PER_FEATURE:
            dropped.append((c, "nan_frac", nan_frac)); continue
        if xf.size < 8:
            dropped.append((c, "too_few_finite", xf.size)); continue
        if np.isfinite(var) and var <= MIN_VAR_PER_FEATURE:
            dropped.append((c, "low_var", var)); continue
        keep.append(c)
    return dedupe_keep_order(keep), dropped

# ══════════════════════════════════════════════════════════════
# FIT / PREDICT
# ══════════════════════════════════════════════════════════════

def fit_predict_scores(model, Xtr, ytr, Xte):
    model.fit(Xtr, ytr)
    score = None
    if hasattr(model, "predict_proba"):
        try: score = model.predict_proba(Xte)[:, 1]
        except: pass
    if score is None and hasattr(model, "decision_function"):
        try: score = model.decision_function(Xte)
        except: pass
    yhat = model.predict(Xte) if score is None else (score >= 0.5).astype(int)
    return yhat, score

def get_oof_scores(model, X, y, groups):
    cv_in = make_group_cv(y, groups, n_splits_desired=min(4, N_SPLITS_DESIRED))
    if cv_in is None: return None
    oof = np.full(len(X), np.nan, dtype=float)
    for tr, va in cv_in.split(X, y, groups):
        mdl = clone(model)
        _, sv = fit_predict_scores(mdl, X.iloc[tr], y[tr], X.iloc[va])
        if sv is None: return None
        oof[va] = to_prob_like(sv)
    return None if np.any(~np.isfinite(oof)) else oof

print("Helpers OK")

---
## SECCIÓN 4 — Catálogo de modelos

### Estrategia de modelos para datasets pequeños con grupos

Con ~30 sujetos y cross-validation por grupo, el espacio de features es relativamente
grande respecto al número de muestras. Las estrategias que mejor funcionan son:

1. **RFF + LinearSVC** (nuestro mejor candidato): proyecta al espacio del kernel RBF
   y luego usa un clasificador lineal. Mucho más rápido que SVM exacto y con ACC comparable.
   La clave es explorar bien el grid de gammas.

2. **SelectKBest + RFF + LR**: reduce features antes de expandir con RFF.
   Elimina el ruido de features poco discriminativas antes de la expansión no lineal.

3. **HistGradientBoosting**: maneja NaN nativamente, no requiere escalar,
   y generaliza bien con pocos datos gracias al early stopping.

4. **SVM RBF con búsqueda de C**: el SVM exacto con cross-val interno para C.

5. **LR + LinearSVC estándar**: baseline lineal siempre útil como referencia.

In [ ]:
def make_models():
    """
    Catálogo completo de modelos.
    Todos devuelven probabilidades (predict_proba) o scores calibrables.

    Nomenclatura:
    - LR          = Logistic Regression (baseline lineal)
    - LSVC        = LinearSVC calibrado (rápido, robusto)
    - SVM_RBF_C*  = SVM con kernel RBF y C específico
    - RFF_LR_g*   = RFF con gamma * + LR (V1: solo esto)
    - RFF_SVC_g*  = RFF con gamma * + LinearSVC (NUEVO: mejor que RFF+LR en muchos casos)
    - RFF_SGD_g*  = RFF con gamma * + SGDClassifier (NUEVO: muy rápido, buena regularización)
    - SKB*_RFF_*  = SelectKBest(k=*) + RFF + LR (NUEVO: filtra ruido antes de expandir)
    - ET          = ExtraTrees (sin escalar, robusto a outliers)
    - HGB         = HistGradientBoosting (maneja NaN, early stopping)
    """
    models = {}

    # ── Lineales de referencia ─────────────────────────────────────────────────
    models["LR"] = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
        ("clf", LogisticRegression(
            solver="liblinear", class_weight="balanced",
            C=1.0, max_iter=8000, random_state=SEED))
    ])

    models["LR_C01"] = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
        ("clf", LogisticRegression(
            solver="liblinear", class_weight="balanced",
            C=0.1, max_iter=8000, random_state=SEED))
    ])

    models["LR_C10"] = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
        ("clf", LogisticRegression(
            solver="liblinear", class_weight="balanced",
            C=10.0, max_iter=8000, random_state=SEED))
    ])

    models["LSVC"] = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
        ("clf", CalibratedClassifierCV(
            estimator=LinearSVC(class_weight="balanced",
                                C=1.0, random_state=SEED),
            method="sigmoid", cv=3))
    ])

    # ── SVM RBF con distintos C ────────────────────────────────────────────────
    # C pequeño = más regularización = mejor generalización con pocos datos
    # C grande  = menos regularización = puede sobreajustar
    for C_val in SVM_C_GRID:
        models[f"SVM_RBF_C{C_val}"] = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc",  StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=C_val, gamma="scale",
                class_weight="balanced",
                probability=True, random_state=SEED))
        ])

    # ── RFF + LR (versión V1 ampliada con más gammas) ──────────────────────────
    # RFF = Random Fourier Features: aproxima el kernel RBF en espacio lineal.
    # Gamma baja  → kernel muy suave → bueno si las clases tienen borde difuso
    # Gamma alta  → kernel local     → bueno si las clases forman clusters
    for gamma in RFF_GAMMAS:
        gstr = str(gamma).replace(".", "_")
        models[f"RFF_LR_g{gstr}"] = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc",  StandardScaler()),
            ("rff", RBFSampler(
                gamma=gamma, n_components=RFF_COMPONENTS, random_state=SEED)),
            ("clf", LogisticRegression(
                solver="liblinear", class_weight="balanced",
                C=1.0, max_iter=8000, random_state=SEED))
        ])

    # ── RFF + LinearSVC (NUEVO — generalmente mejor que RFF+LR) ───────────────
    # LinearSVC en espacio RFF = mejor margen en el espacio del kernel
    # Es el clasificador estándar para kernels aproximados en la literatura
    for gamma in RFF_GAMMAS:
        gstr = str(gamma).replace(".", "_")
        models[f"RFF_SVC_g{gstr}"] = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc",  StandardScaler()),
            ("rff", RBFSampler(
                gamma=gamma, n_components=RFF_COMPONENTS, random_state=SEED)),
            ("clf", CalibratedClassifierCV(
                estimator=LinearSVC(
                    class_weight="balanced", C=1.0, random_state=SEED),
                method="sigmoid", cv=3))
        ])

    # ── RFF + SGD (NUEVO — muy rápido, buena regularización L2) ───────────────
    # SGD con loss=modified_huber da probabilidades y converge rápido
    for gamma in [0.25, 0.5, 1.0, 2.0]:  # subset de gammas para no explotar tiempo
        gstr = str(gamma).replace(".", "_")
        models[f"RFF_SGD_g{gstr}"] = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc",  StandardScaler()),
            ("rff", RBFSampler(
                gamma=gamma, n_components=RFF_COMPONENTS, random_state=SEED)),
            ("clf", SGDClassifier(
                loss="modified_huber", penalty="l2",
                class_weight="balanced", random_state=SEED,
                max_iter=2000, tol=1e-4))
        ])

    # ── SelectKBest + RFF + LR (NUEVO — filtra ruido antes de expandir) ────────
    # Si hay features poco discriminativas, RFF las amplifica como ruido.
    # SelectKBest elimina las peores features antes de la expansión no lineal.
    for k in [v for v in SKB_K_VALUES if v != "all"]:
        for gamma in [0.25, 0.5, 1.0]:
            gstr = str(gamma).replace(".", "_")
            models[f"SKB{k}_RFF_LR_g{gstr}"] = Pipeline([
                ("imp",  SimpleImputer(strategy="median")),
                ("sc",   StandardScaler()),
                ("skb",  SelectKBest(f_classif, k=k)),
                ("rff",  RBFSampler(
                    gamma=gamma, n_components=RFF_COMPONENTS, random_state=SEED)),
                ("clf",  LogisticRegression(
                    solver="liblinear", class_weight="balanced",
                    C=1.0, max_iter=8000, random_state=SEED))
            ])

    # ── SelectKBest + RFF + SVC (NUEVO — la combinación más potente) ──────────
    for k in [6, 10]:
        for gamma in [0.5, 1.0]:
            gstr = str(gamma).replace(".", "_")
            models[f"SKB{k}_RFF_SVC_g{gstr}"] = Pipeline([
                ("imp",  SimpleImputer(strategy="median")),
                ("sc",   StandardScaler()),
                ("skb",  SelectKBest(f_classif, k=k)),
                ("rff",  RBFSampler(
                    gamma=gamma, n_components=RFF_COMPONENTS, random_state=SEED)),
                ("clf",  CalibratedClassifierCV(
                    estimator=LinearSVC(
                        class_weight="balanced", C=1.0, random_state=SEED),
                    method="sigmoid", cv=3))
            ])

    # ── Tree ensembles ─────────────────────────────────────────────────────────
    models["ET"] = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", ExtraTreesClassifier(
            n_estimators=1500, random_state=SEED,
            class_weight="balanced", n_jobs=-1,
            min_samples_leaf=2, max_features="sqrt"))
    ])

    # HistGradientBoosting: maneja NaN nativamente, muy bueno con pocos datos
    # class_weight no existe → usamos sample_weight internamente via class_weight
    models["HGB"] = Pipeline([
        ("clf", HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.05,
            max_depth=3, min_samples_leaf=5,
            random_state=SEED,
            early_stopping=True, n_iter_no_change=20,
            class_weight="balanced"))
    ])

    models["HGB_lr01"] = Pipeline([
        ("clf", HistGradientBoostingClassifier(
            max_iter=500, learning_rate=0.01,
            max_depth=4, min_samples_leaf=4,
            random_state=SEED,
            early_stopping=True, n_iter_no_change=30,
            class_weight="balanced"))
    ])

    total = len(models)
    print(f"Catálogo de modelos: {total} modelos")
    print("  LR variants:       ", len([k for k in models if k.startswith("LR")]))
    print("  SVM RBF variants:  ", len([k for k in models if k.startswith("SVM")]))
    print("  RFF_LR variants:   ", len([k for k in models if k.startswith("RFF_LR")]))
    print("  RFF_SVC variants:  ", len([k for k in models if k.startswith("RFF_SVC")]))
    print("  RFF_SGD variants:  ", len([k for k in models if k.startswith("RFF_SGD")]))
    print("  SKB+RFF variants:  ", len([k for k in models if k.startswith("SKB")]))
    print("  Tree ensembles:    ", len([k for k in models if k in ["ET","HGB","HGB_lr01"]]))
    return models

# Probar que se construyen bien
_ = make_models()
print("✅ Modelos OK")

---
## SECCIÓN 5 — Carga y preparación de datos

In [ ]:
def _rename_non_keys(df, key_cols, prefix):
    return df.rename(columns={c: f"{prefix}{c}" for c in df.columns if c not in key_cols})

def _drop_h2_extra(df):
    return df.drop(
        columns=[c for c in ["pipeline_variant","group","kind","npz_path","ruta_npz"]
                 if c in df.columns]
    )

def _filter_h2(df, pipeline_variant):
    col_pv = "pipeline_variant" if "pipeline_variant" in df.columns else None
    if col_pv:
        df = df[df[col_pv].astype(str) == str(pipeline_variant)].copy()
    if "kind" in df.columns:
        df = df[df["kind"].astype(str) == "all"].copy()
    return df

def prep_h1_bywin(df):
    keys = [c for c in ["id","y","cond","win_sec","tercile"] if c in df.columns]
    return _rename_non_keys(df.copy(), keys, "h1__")

def prep_h1_pooled(df):
    keys = [c for c in ["id","y","cond","tercile"] if c in df.columns]
    return _rename_non_keys(df.copy(), keys, "h1__")

def prep_h1_delta(df):
    keys = [c for c in ["id","y","win_sec","tercile"] if c in df.columns]
    return _rename_non_keys(df.copy(), keys, "h1__")

def prep_h2_bywin(df, pv):
    keys = [c for c in ["id","y","cond","win_sec","tercile"] if c in df.columns]
    return _rename_non_keys(_drop_h2_extra(_filter_h2(df, pv)), keys, "h2__")

def prep_h2_pooled(df, pv):
    keys = [c for c in ["id","y","cond","tercile"] if c in df.columns]
    df2  = _filter_h2(df, pv)
    df2  = df2.drop(
        columns=[c for c in ["pipeline_variant","group","kind","win_sec","npz_path","ruta_npz"]
                 if c in df2.columns]
    )
    return _rename_non_keys(df2, keys, "h2__")

def prep_h2_delta(df, pv):
    keys = [c for c in ["id","y","win_sec","tercile"] if c in df.columns]
    df2  = _filter_h2(df, pv)
    df2  = df2.drop(
        columns=[c for c in ["pipeline_variant","group","kind","cond","npz_path","ruta_npz"]
                 if c in df2.columns]
    )
    return _rename_non_keys(df2, keys, "h2__")

def load_all_data():
    required = [GSG_BYWIN, GSG_POOLED, GSG_DELTA_BYWIN, DMSD_BYWIN, DMSD_POOLED, DMSD_DELTA_BYWIN]
    missing  = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Faltan archivos:\n" + "\n".join(missing))

    data = dict(
        h1_bywin  = pd.read_csv(GSG_BYWIN),
        h1_pooled = pd.read_csv(GSG_POOLED),
        h1_delta  = pd.read_csv(GSG_DELTA_BYWIN),
        h2_bywin  = pd.read_csv(DMSD_BYWIN),
        h2_pooled = pd.read_csv(DMSD_POOLED),
        h2_delta  = pd.read_csv(DMSD_DELTA_BYWIN),
    )
    print("Datos cargados:")
    for k, v in data.items():
        print(f"  {k:15s} → {v.shape}")
    return data

print("Funciones de prep OK")

---
## SECCIÓN 6 — EARLY FUSION

Fusiona features de global spectral geometry y dominant-mode spatial distribution **antes** de entrenar el modelo.

**Ventaja**: el modelo ve la interacción entre features global spectral geometry y dominant-mode spatial distribution directamente.

**Nuevos feature sets:**
- `h1_core__h2_full`: versión mínima global spectral geometry + todas las features dominant-mode spatial distribution
- `h1_priority__h2_full`: global spectral geometry completo + dominant-mode spatial distribution completo
- `h1_delta__h2_delta_full`: deltas global spectral geometry + deltas dominant-mode spatial distribution ampliados

In [ ]:
def build_early_fusion_sets_static(df):
    """
    Feature sets para datos estáticos (bywin y pooled).
    Cada combinación global spectral geometry × dominant-mode spatial distribution es un experimento diferente.
    """
    sets = {
        # V1 sets (mantener para comparación)
        "h1_priority__h2_distribution":
            available_features(df,
                [f"h1__{x}" for x in GSG_PRIORITY] +
                [f"h2__{x}" for x in DMSD_DISTRIBUTION]),
        "h1_priority__h2_priority_stat":
            available_features(df,
                [f"h1__{x}" for x in GSG_PRIORITY] +
                [f"h2__{x}" for x in DMSD_PRIORITY_STAT]),
        "h1_statcompact__h2_distribution":
            available_features(df,
                [f"h1__{x}" for x in GSG_STAT_COMPACT] +
                [f"h2__{x}" for x in DMSD_DISTRIBUTION]),
        "h1_statcompact__h2_priority_noocc":
            available_features(df,
                [f"h1__{x}" for x in GSG_STAT_COMPACT] +
                [f"h2__{x}" for x in DMSD_PRIORITY_NO_OCCIPITAL_VISUAL]),
        "h1_priority__h2_priority_noocc":
            available_features(df,
                [f"h1__{x}" for x in GSG_PRIORITY] +
                [f"h2__{x}" for x in DMSD_PRIORITY_NO_OCCIPITAL_VISUAL]),
        # NUEVOS: versiones ampliadas
        "h1_core__h2_full":
            available_features(df,
                [f"h1__{x}" for x in GSG_CORE_ONLY] +
                [f"h2__{x}" for x in DMSD_FULL]),
        "h1_priority__h2_full":
            available_features(df,
                [f"h1__{x}" for x in GSG_PRIORITY] +
                [f"h2__{x}" for x in DMSD_FULL]),
        "h1_statcompact__h2_full":
            available_features(df,
                [f"h1__{x}" for x in GSG_STAT_COMPACT] +
                [f"h2__{x}" for x in DMSD_FULL]),
    }
    return {k: v for k, v in sets.items() if len(v) >= MIN_FEATURES_AFTER_CLEAN}

def build_early_fusion_sets_delta(df):
    """
    Feature sets para datos delta (cambio BASAL→PVT).
    """
    sets = {
        # V1
        "h1_deltafocus__h2_deltapriority":
            available_features(df,
                [f"h1__{x}" for x in GSG_DELTA_FOCUS] +
                [f"h2__{x}" for x in DMSD_PRIORITY_DELTA]),
        "h1_deltafocus__h2_deltapriority_reorg":
            available_features(df,
                [f"h1__{x}" for x in GSG_DELTA_FOCUS] +
                [f"h2__{x}" for x in (DMSD_PRIORITY_DELTA + DMSD_DELTA_REORG)]),
        # NUEVOS
        "h1_deltafocus__h2_delta_full":
            available_features(df,
                [f"h1__{x}" for x in GSG_DELTA_FOCUS] +
                [f"h2__{x}" for x in DMSD_DELTA_FULL]),
    }
    return {k: v for k, v in sets.items() if len(v) >= MIN_FEATURES_AFTER_CLEAN}


def _early_evaluate_block(df_block, feature_cols, tag, fs_name):
    """
    Evalúa todos los modelos del catálogo sobre un bloque con CV.
    Gestiona el caso de SKB con k > n_features_disponibles.
    """
    feature_cols = dedupe_keep_order([c for c in feature_cols if c in df_block.columns])
    if len(feature_cols) < MIN_FEATURES_AFTER_CLEAN:
        return pd.DataFrame()

    X      = build_X_numeric(df_block, feature_cols)
    y      = df_block["y"].astype(int).values
    groups = df_block["id"].astype(str).values

    tmp = pd.DataFrame({"g": groups, "y": y}).drop_duplicates("g")
    if tmp["y"].nunique() < 2: return pd.DataFrame()
    if (int((tmp["y"]==0).sum()) < MIN_GROUPS_PER_CLASS or
        int((tmp["y"]==1).sum()) < MIN_GROUPS_PER_CLASS):
        return pd.DataFrame()

    cv = make_group_cv(y, groups, n_splits_desired=N_SPLITS_DESIRED)
    if cv is None: return pd.DataFrame()

    cleaned_cols, _ = clean_feature_block(df_block, feature_cols)
    if len(cleaned_cols) < MIN_FEATURES_AFTER_CLEAN:
        return pd.DataFrame()
    X = X[cleaned_cols].copy()
    n_feats_available = len(cleaned_cols)

    rows = []

    for model_name, model_template in make_models().items():
        # Ajustar k de SKB si es mayor que features disponibles
        model = clone(model_template)
        if "skb" in model.named_steps:
            k_req = model.named_steps["skb"].k
            if isinstance(k_req, int) and k_req > n_feats_available:
                continue  # saltar este modelo para este bloque

        bal_list, f1_list, auc_list = [], [], []
        sbal_list, sf1_list, sauc_list = [], [], []
        subj_n_test, thr_list = [], []

        fold_ok = 0
        for tr, te in cv.split(X, y, groups):
            Xtr, Xte = X.iloc[tr].copy(), X.iloc[te].copy()
            ytr, yte = y[tr], y[te]
            gtr, gte = groups[tr], groups[te]

            oof_tr = get_oof_scores(clone(model), Xtr, ytr, gtr)
            thr = choose_best_threshold(ytr, oof_tr)[0] if oof_tr is not None else 0.5
            thr_list.append(thr)

            _, score_te = fit_predict_scores(clone(model), Xtr, ytr, Xte)
            if score_te is None: continue

            score_te = to_prob_like(score_te)
            mets = evaluate_scores(yte, gte, score_te, thr)

            bal_list.append(mets["bal_acc"])
            f1_list.append(mets["f1_macro"])
            auc_list.append(mets["auc"])
            sbal_list.append(mets["subj_bal_acc"])
            sf1_list.append(mets["subj_f1_macro"])
            sauc_list.append(mets["subj_auc"])
            subj_n_test.append(mets["subj_n_test"])
            fold_ok += 1

        if fold_ok == 0: continue

        rows.append(dict(
            tag=tag, feature_set=fs_name, model=model_name,
            n_features=n_feats_available, n_folds=fold_ok,
            n_subjects=int(tmp["y"].count()),
            bal_acc_mean=float(np.nanmean(bal_list)),
            bal_acc_std=float(np.nanstd(bal_list, ddof=1)) if len(bal_list)>1 else np.nan,
            f1_macro_mean=float(np.nanmean(f1_list)),
            f1_macro_std=float(np.nanstd(f1_list, ddof=1)) if len(f1_list)>1 else np.nan,
            auc_mean=float(np.nanmean(auc_list)),
            auc_std=float(np.nanstd(auc_list, ddof=1)) if len(auc_list)>1 else np.nan,
            subj_bal_acc_mean=float(np.nanmean(sbal_list)),
            subj_f1_macro_mean=float(np.nanmean(sf1_list)),
            subj_auc_mean=float(np.nanmean(sauc_list)),
            subj_n_test_mean=float(np.nanmean(subj_n_test)),
            threshold_mean=float(np.nanmean(thr_list)),
        ))

    return pd.DataFrame(rows)


def _early_run(df_h1, df_h2, merge_keys, groupby_cols, tag_prefix, build_sets_fn):
    dfm = df_h1.merge(df_h2, on=merge_keys, how="inner")
    if dfm.empty: return pd.DataFrame()

    results = []
    for keys, d in dfm.groupby(groupby_cols, dropna=False):
        if len(d) < MIN_ROWS_PER_BLOCK or d["id"].nunique() < MIN_SUBJECTS_PER_BLOCK:
            continue
        tag  = tag_prefix + "__" + "__".join(
            str(k) for k in (keys if isinstance(keys, tuple) else (keys,)))
        sets = build_sets_fn(d)
        for fs_name, fs_cols in sets.items():
            r = _early_evaluate_block(d, fs_cols, tag=f"{tag}__{fs_name}", fs_name=fs_name)
            if not r.empty: results.append(r)

    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()


def run_early_bywin(df_h1, df_h2, pv):
    return _early_run(
        prep_h1_bywin(df_h1), prep_h2_bywin(df_h2, pv),
        merge_keys=["id","y","cond","win_sec","tercile"],
        groupby_cols=["cond","win_sec","tercile"],
        tag_prefix=f"early_bywin__{pv}",
        build_sets_fn=build_early_fusion_sets_static)

def run_early_pooled(df_h1, df_h2, pv):
    return _early_run(
        prep_h1_pooled(df_h1), prep_h2_pooled(df_h2, pv),
        merge_keys=["id","y","cond","tercile"],
        groupby_cols=["cond","tercile"],
        tag_prefix=f"early_pooled__{pv}",
        build_sets_fn=build_early_fusion_sets_static)

def run_early_delta(df_h1, df_h2, pv):
    return _early_run(
        prep_h1_delta(df_h1), prep_h2_delta(df_h2, pv),
        merge_keys=["id","y","win_sec","tercile"],
        groupby_cols=["win_sec","tercile"],
        tag_prefix=f"early_delta__{pv}",
        build_sets_fn=build_early_fusion_sets_delta)


def save_early_summary(df_res, out_prefix):
    if df_res is None or df_res.empty: return
    sort_cols = ["subj_f1_macro_mean","subj_bal_acc_mean","subj_auc_mean",
                 "f1_macro_mean","bal_acc_mean","auc_mean"]
    df_sorted = df_res.sort_values(sort_cols, ascending=[False]*len(sort_cols))

    # Mejor por (tag, feature_set)
    (df_sorted.groupby(["tag","feature_set"], as_index=False).head(1)
     .to_csv(OUT_DIR_EARLY / f"{out_prefix}_best_per_tag_fs.csv",
             index=False, float_format=CSV_FLOAT_FMT))

    # Top-1 por tag (mejor modelo en cualquier feature_set)
    (df_sorted.groupby("tag", as_index=False).head(1)
     .to_csv(OUT_DIR_EARLY / f"{out_prefix}_top1_per_tag.csv",
             index=False, float_format=CSV_FLOAT_FMT))

    # Media por modelo (para saber qué familia funciona mejor)
    (df_res.groupby("model", as_index=False)[sort_cols].mean(numeric_only=True)
     .sort_values(sort_cols[:2], ascending=[False, False])
     .to_csv(OUT_DIR_EARLY / f"{out_prefix}_mean_by_model.csv",
             index=False, float_format=CSV_FLOAT_FMT))

print("Early Fusion OK")

---
## SECCIÓN 7 — LATE FUSION

Combina las probabilidades de modelos global spectral geometry y dominant-mode spatial distribution **después** de entrenarlos por separado.

**Mejoras respecto a V1:**
- Meta-stacking con múltiples features (no solo p_h1, p_h2 sino también diferencia, producto)
- Meta-modelos adicionales: RidgeClassifier calibrado, SGD
- El `select_best_base_model` usa el catálogo completo de V2

In [ ]:
def build_late_h1_static(df):
    return {k: v for k, v in {
        "h1_priority":   available_features(df, [f"h1__{x}" for x in GSG_PRIORITY]),
        "h1_statcompact":available_features(df, [f"h1__{x}" for x in GSG_STAT_COMPACT]),
        "h1_core":       available_features(df, [f"h1__{x}" for x in GSG_CORE_ONLY]),
    }.items() if len(v) >= MIN_FEATURES_AFTER_CLEAN}

def build_late_h1_delta(df):
    return {k: v for k, v in {
        "h1_delta_focus": available_features(df, [f"h1__{x}" for x in GSG_DELTA_FOCUS]),
    }.items() if len(v) >= MIN_FEATURES_AFTER_CLEAN}

def build_late_h2_static(df):
    return {k: v for k, v in {
        "h2_distribution":    available_features(df, [f"h2__{x}" for x in DMSD_DISTRIBUTION]),
        "h2_priority_stat":   available_features(df, [f"h2__{x}" for x in DMSD_PRIORITY_STAT]),
        "h2_priority_noocc":  available_features(df, [f"h2__{x}" for x in DMSD_PRIORITY_NO_OCCIPITAL_VISUAL]),
        "h2_full":            available_features(df, [f"h2__{x}" for x in DMSD_FULL]),
    }.items() if len(v) >= MIN_FEATURES_AFTER_CLEAN}

def build_late_h2_delta(df):
    return {k: v for k, v in {
        "h2_delta_priority":  available_features(df, [f"h2__{x}" for x in DMSD_PRIORITY_DELTA]),
        "h2_delta_reorg":     available_features(df, [f"h2__{x}" for x in DMSD_DELTA_REORG]),
        "h2_delta_full":      available_features(df, [f"h2__{x}" for x in DMSD_DELTA_FULL]),
    }.items() if len(v) >= MIN_FEATURES_AFTER_CLEAN}


def select_best_base_model(X, y, groups, candidate_feature_sets):
    """Elige el mejor (feature_set, modelo) mediante OOF dentro del fold de train."""
    candidates = []
    for fs_name, fs_cols in candidate_feature_sets.items():
        fs_cols = dedupe_keep_order([c for c in fs_cols if c in X.columns])
        if len(fs_cols) < MIN_FEATURES_AFTER_CLEAN: continue
        cleaned, dropped = clean_feature_block(X[fs_cols], fs_cols)
        if len(cleaned) < MIN_FEATURES_AFTER_CLEAN: continue
        Xsub = build_X_numeric(X, cleaned)
        n_feats = len(cleaned)

        for model_name, model_template in make_models().items():
            model = clone(model_template)
            # Saltar SKB con k > features disponibles
            if "skb" in getattr(model, "named_steps", {}):
                k_req = model.named_steps["skb"].k
                if isinstance(k_req, int) and k_req > n_feats:
                    continue

            oof = get_oof_scores(model, Xsub, y, groups)
            if oof is None: continue
            thr, _ = choose_best_threshold(y, oof)
            mets   = evaluate_scores(y, groups, oof, thr)
            candidates.append(dict(
                feature_set=fs_name, model_name=model_name,
                features_used=cleaned, model=model,
                threshold=thr, n_features=n_feats,
                n_features_dropped=len(dropped),
                oof_subj_f1=mets["subj_f1_macro"],
                oof_subj_bal=mets["subj_bal_acc"],
                oof_subj_auc=mets["subj_auc"],
                oof_row_f1=mets["f1_macro"],
                oof_row_auc=mets["auc"],
            ))

    if not candidates: return None
    return (pd.DataFrame(candidates)
              .sort_values(
                  ["oof_subj_f1","oof_subj_bal","oof_subj_auc","oof_row_f1","oof_row_auc"],
                  ascending=False)
              .iloc[0].to_dict())


def _make_meta_features(p_h1, p_h2):
    """
    Construye un DataFrame de meta-features para el stacking.
    Más features para el meta-modelo = mejor calibración del ensemble.

    Features:
    - p_h1, p_h2: probabilidades base
    - diff: diferencia (indica si global spectral geometry y dominant-mode spatial distribution están de acuerdo)
    - prod: producto (señal conjunta de ambos)
    - max_p, min_p: extremos (detecta cuando uno de los dos es muy confiante)
    """
    return pd.DataFrame({
        "p_h1":  p_h1,
        "p_h2":  p_h2,
        "diff":  p_h1 - p_h2,
        "prod":  p_h1 * p_h2,
        "max_p": np.maximum(p_h1, p_h2),
        "min_p": np.minimum(p_h1, p_h2),
    })


def _late_evaluate_block(df_h1_block, df_h2_block, tag, scope_label, pv):
    if "id" not in df_h1_block.columns or "y" not in df_h1_block.columns:
        return pd.DataFrame()

    ids = df_h1_block["id"].astype(str).values
    y   = df_h1_block["y"].astype(int).values

    tmp = pd.DataFrame({"g": ids, "y": y}).drop_duplicates("g")
    if tmp["y"].nunique() < 2: return pd.DataFrame()
    if (int((tmp["y"]==0).sum()) < MIN_GROUPS_PER_CLASS or
        int((tmp["y"]==1).sum()) < MIN_GROUPS_PER_CLASS):
        return pd.DataFrame()

    cv = make_group_cv(y, ids, n_splits_desired=N_SPLITS_DESIRED)
    if cv is None: return pd.DataFrame()

    is_delta = ("cond" not in df_h1_block.columns) and ("win_sec" in df_h1_block.columns)
    h1_sets  = build_late_h1_delta(df_h1_block) if is_delta else build_late_h1_static(df_h1_block)
    h2_sets  = build_late_h2_delta(df_h2_block) if is_delta else build_late_h2_static(df_h2_block)

    meta_cols    = {"id","y","cond","win_sec","tercile"}
    h1_feat_cols = [c for c in df_h1_block.columns if c not in meta_cols and c.startswith("h1__")]
    h2_feat_cols = [c for c in df_h2_block.columns if c not in meta_cols and c.startswith("h2__")]

    Xh1_all = df_h1_block[h1_feat_cols].copy()
    Xh2_all = df_h2_block[h2_feat_cols].copy()

    fold_rows = []

    for fold_idx, (tr, te) in enumerate(cv.split(np.zeros(len(y)), y, ids), 1):
        ids_tr, ids_te = ids[tr], ids[te]
        ytr, yte = y[tr], y[te]

        best_h1 = select_best_base_model(Xh1_all.iloc[tr].copy(), ytr, ids_tr, h1_sets)
        if best_h1 is None: continue
        best_h2 = select_best_base_model(Xh2_all.iloc[tr].copy(), ytr, ids_tr, h2_sets)
        if best_h2 is None: continue

        # Ajustar global spectral geometry
        Xh1_tr = build_X_numeric(Xh1_all.iloc[tr], best_h1["features_used"])
        Xh1_te = build_X_numeric(Xh1_all.iloc[te], best_h1["features_used"])
        _, s_h1_te = fit_predict_scores(clone(best_h1["model"]), Xh1_tr, ytr, Xh1_te)
        if s_h1_te is None: continue
        s_h1_te = to_prob_like(s_h1_te)
        oof_h1  = get_oof_scores(clone(best_h1["model"]), Xh1_tr, ytr, ids_tr)
        if oof_h1 is None: continue

        # Ajustar dominant-mode spatial distribution
        Xh2_tr = build_X_numeric(Xh2_all.iloc[tr], best_h2["features_used"])
        Xh2_te = build_X_numeric(Xh2_all.iloc[te], best_h2["features_used"])
        _, s_h2_te = fit_predict_scores(clone(best_h2["model"]), Xh2_tr, ytr, Xh2_te)
        if s_h2_te is None: continue
        s_h2_te = to_prob_like(s_h2_te)
        oof_h2  = get_oof_scores(clone(best_h2["model"]), Xh2_tr, ytr, ids_tr)
        if oof_h2 is None: continue

        base_info = dict(
            tag=tag, scope=scope_label, pipeline_variant=pv, fold=fold_idx,
            base_h1_feature_set=best_h1["feature_set"], base_h1_model=best_h1["model_name"],
            base_h2_feature_set=best_h2["feature_set"], base_h2_model=best_h2["model_name"],
        )

        # global spectral geometry solo
        fold_rows.append({**base_info, "ensemble":"h1_only", "weight_h1":np.nan,
                          "threshold":best_h1["threshold"],
                          **evaluate_scores(yte, ids_te, s_h1_te, best_h1["threshold"])})
        # dominant-mode spatial distribution solo
        fold_rows.append({**base_info, "ensemble":"h2_only", "weight_h1":np.nan,
                          "threshold":best_h2["threshold"],
                          **evaluate_scores(yte, ids_te, s_h2_te, best_h2["threshold"])})

        # Blends ponderados
        for w in BLEND_WEIGHTS_GSG:
            blend_tr = w * oof_h1 + (1.0-w) * oof_h2
            thr_blend, _ = choose_best_threshold(ytr, blend_tr)
            blend_te = w * s_h1_te + (1.0-w) * s_h2_te
            fold_rows.append({**base_info, "ensemble":"blend_weighted", "weight_h1":w,
                              "threshold":thr_blend,
                              **evaluate_scores(yte, ids_te, blend_te, thr_blend)})

        # ── META-STACKING con features enriquecidas ────────────────────────────
        # Construir meta-features de train
        meta_Xtr = _make_meta_features(oof_h1, oof_h2)
        meta_Xte = _make_meta_features(s_h1_te, s_h2_te)

        for meta_name, meta_clf in [
            # LR como en V1 (baseline del stacking)
            ("stacking_LR",
             LogisticRegression(solver="liblinear", class_weight="balanced",
                                C=1.0, max_iter=8000, random_state=SEED)),
            # LR con más regularización (evita sobreajustar el meta-modelo)
            ("stacking_LR_C01",
             LogisticRegression(solver="liblinear", class_weight="balanced",
                                C=0.1, max_iter=8000, random_state=SEED)),
            # Ridge calibrado: lineal pero con regularización L2
            ("stacking_Ridge",
             CalibratedClassifierCV(
                 estimator=RidgeClassifier(alpha=1.0, class_weight="balanced"),
                 method="sigmoid", cv=3)),
            # SGD: muy rápido, buena regularización
            ("stacking_SGD",
             SGDClassifier(loss="modified_huber", penalty="l2", alpha=0.01,
                           class_weight="balanced", random_state=SEED,
                           max_iter=2000, tol=1e-4)),
        ]:
            try:
                meta_clf.fit(meta_Xtr, ytr)
                if hasattr(meta_clf, "predict_proba"):
                    s_meta_tr  = meta_clf.predict_proba(meta_Xtr)[:, 1]
                    s_meta_te  = meta_clf.predict_proba(meta_Xte)[:, 1]
                else:
                    s_meta_tr  = meta_clf.decision_function(meta_Xtr)
                    s_meta_te  = meta_clf.decision_function(meta_Xte)
                    s_meta_tr  = to_prob_like(s_meta_tr)
                    s_meta_te  = to_prob_like(s_meta_te)

                thr_meta, _ = choose_best_threshold(ytr, s_meta_tr)
                fold_rows.append({
                    **base_info, "ensemble": meta_name, "weight_h1": np.nan,
                    "threshold": thr_meta,
                    **evaluate_scores(yte, ids_te, s_meta_te, thr_meta)
                })
            except Exception:
                pass

    if not fold_rows: return pd.DataFrame()

    dff = pd.DataFrame(fold_rows)
    group_cols = ["tag","scope","pipeline_variant","ensemble","weight_h1"]
    summary = []
    for keys, g in dff.groupby(group_cols, dropna=False):
        summary.append({
            "tag":keys[0], "scope":keys[1], "pipeline_variant":keys[2],
            "ensemble":keys[3], "weight_h1":keys[4],
            "n_folds":               int(len(g)),
            "bal_acc_mean":          float(np.nanmean(g["bal_acc"])),
            "bal_acc_std":           float(np.nanstd(g["bal_acc"],ddof=1)) if len(g)>1 else np.nan,
            "f1_macro_mean":         float(np.nanmean(g["f1_macro"])),
            "f1_macro_std":          float(np.nanstd(g["f1_macro"],ddof=1)) if len(g)>1 else np.nan,
            "auc_mean":              float(np.nanmean(g["auc"])),
            "auc_std":               float(np.nanstd(g["auc"],ddof=1)) if len(g)>1 else np.nan,
            "subj_bal_acc_mean":     float(np.nanmean(g["subj_bal_acc"])),
            "subj_bal_acc_std":      float(np.nanstd(g["subj_bal_acc"],ddof=1)) if len(g)>1 else np.nan,
            "subj_f1_macro_mean":    float(np.nanmean(g["subj_f1_macro"])),
            "subj_f1_macro_std":     float(np.nanstd(g["subj_f1_macro"],ddof=1)) if len(g)>1 else np.nan,
            "subj_auc_mean":         float(np.nanmean(g["subj_auc"])),
            "subj_auc_std":          float(np.nanstd(g["subj_auc"],ddof=1)) if len(g)>1 else np.nan,
            "threshold_mean":        float(np.nanmean(g["threshold"])),
            "threshold_std":         float(np.nanstd(g["threshold"],ddof=1)) if len(g)>1 else np.nan,
            "base_h1_feature_set":   most_frequent_nonempty(g["base_h1_feature_set"]),
            "base_h1_model":         most_frequent_nonempty(g["base_h1_model"]),
            "base_h2_feature_set":   most_frequent_nonempty(g["base_h2_feature_set"]),
            "base_h2_model":         most_frequent_nonempty(g["base_h2_model"]),
        })
    return pd.DataFrame(summary)


def _late_run(h1, h2, merge_keys, groupby_cols, tag_prefix, scope_label, pv):
    dfm = h1.merge(h2, on=merge_keys, how="inner")
    if dfm.empty: return pd.DataFrame()

    h1_cols = [c for c in h1.columns if c not in merge_keys]
    h2_cols = [c for c in h2.columns if c not in merge_keys]
    results = []

    for keys, d in dfm.groupby(groupby_cols, dropna=False):
        if len(d) < MIN_ROWS_PER_BLOCK or d["id"].nunique() < MIN_SUBJECTS_PER_BLOCK:
            continue
        tag = tag_prefix + "__" + "__".join(
            str(k) for k in (keys if isinstance(keys, tuple) else (keys,)))
        out = _late_evaluate_block(
            d[merge_keys + h1_cols].copy(),
            d[merge_keys + h2_cols].copy(),
            tag=tag, scope_label=scope_label, pv=pv)
        if not out.empty: results.append(out)

    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()


def run_late_bywin(df_h1, df_h2, pv):
    return _late_run(
        prep_h1_bywin(df_h1), prep_h2_bywin(df_h2, pv),
        merge_keys=["id","y","cond","win_sec","tercile"],
        groupby_cols=["cond","win_sec","tercile"],
        tag_prefix=f"late_bywin__{pv}", scope_label="bywin", pv=pv)

def run_late_pooled(df_h1, df_h2, pv):
    return _late_run(
        prep_h1_pooled(df_h1), prep_h2_pooled(df_h2, pv),
        merge_keys=["id","y","cond","tercile"],
        groupby_cols=["cond","tercile"],
        tag_prefix=f"late_pooled__{pv}", scope_label="pooled", pv=pv)

def run_late_delta(df_h1, df_h2, pv):
    return _late_run(
        prep_h1_delta(df_h1), prep_h2_delta(df_h2, pv),
        merge_keys=["id","y","win_sec","tercile"],
        groupby_cols=["win_sec","tercile"],
        tag_prefix=f"late_delta__{pv}", scope_label="delta_bywin", pv=pv)


def save_late_summary(df_res, out_prefix):
    if df_res is None or df_res.empty: return
    sort_cols = ["subj_f1_macro_mean","subj_bal_acc_mean","subj_auc_mean","f1_macro_mean"]
    valid = df_res[df_res["n_folds"] >= MIN_VALID_FOLDS_FOR_MAIN_RANKING].copy()

    if not valid.empty:
        (valid.sort_values(["tag"]+sort_cols, ascending=[True]+[False]*4)
              .groupby("tag", as_index=False).head(1)
              .to_csv(OUT_DIR_LATE / f"{out_prefix}_best_by_tag_valid.csv",
                      index=False, float_format=CSV_FLOAT_FMT))

    (df_res.sort_values(["tag"]+sort_cols, ascending=[True]+[False]*4)
           .groupby("tag", as_index=False).head(1)
           .to_csv(OUT_DIR_LATE / f"{out_prefix}_best_by_tag_all.csv",
                   index=False, float_format=CSV_FLOAT_FMT))

    (df_res.groupby("ensemble", as_index=False)[sort_cols].mean(numeric_only=True)
           .sort_values(sort_cols[:2], ascending=[False,False])
           .to_csv(OUT_DIR_LATE / f"{out_prefix}_mean_by_ensemble.csv",
                   index=False, float_format=CSV_FLOAT_FMT))

print("Late Fusion OK")

---
## SECCIÓN 8 — Main unificado

In [ ]:
def main():
    import time
    t0 = time.time()

    print("=" * 62)
    print("combined eigenmode representations  EARLY & LATE FUSION V2")
    print(f"  EARLY → {OUT_DIR_EARLY}")
    print(f"  LATE  → {OUT_DIR_LATE}")
    n_modelos = len(make_models())
    print(f"  Modelos en catálogo: {n_modelos}")
    print("=" * 62)

    DATA = load_all_data()

    early_all = []
    late_all  = []

    for pv in DMSD_PIPELINES:
        print(f"\n{'─'*55}")
        print(f"PIPELINE dominant-mode spatial distribution: {pv}")
        print(f"{'─'*55}")

        # ── EARLY FUSION ──────────────────────────────────────────
        for scope, runner, h1key, h2key in [
            ("bywin",  run_early_bywin,  "h1_bywin",  "h2_bywin"),
            ("pooled", run_early_pooled, "h1_pooled", "h2_pooled"),
            ("delta",  run_early_delta,  "h1_delta",  "h2_delta"),
        ]:
            print(f"  [EARLY] {scope}...")
            r = runner(DATA[h1key], DATA[h2key], pv)
            if not r.empty:
                r.to_csv(OUT_DIR_EARLY / f"early_{scope}__{pv}.csv",
                         index=False, float_format=CSV_FLOAT_FMT)
                save_early_summary(r, f"early_{scope}__{pv}")
                early_all.append(r.assign(scope=scope, pipeline_variant=pv))
                print(f"     → {len(r)} filas guardadas")
            else:
                print("     → sin resultados")

        # ── LATE FUSION ───────────────────────────────────────────
        for scope, runner, h1key, h2key in [
            ("bywin",  run_late_bywin,  "h1_bywin",  "h2_bywin"),
            ("pooled", run_late_pooled, "h1_pooled", "h2_pooled"),
            ("delta",  run_late_delta,  "h1_delta",  "h2_delta"),
        ]:
            print(f"  [LATE]  {scope}...")
            r = runner(DATA[h1key], DATA[h2key], pv)
            if not r.empty:
                r.to_csv(OUT_DIR_LATE / f"late_{scope}__{pv}.csv",
                         index=False, float_format=CSV_FLOAT_FMT)
                save_late_summary(r, f"late_{scope}__{pv}")
                late_all.append(r.assign(global_scope=scope))
                print(f"     → {len(r)} filas guardadas")
            else:
                print("     → sin resultados")

    # ── Resúmenes globales ─────────────────────────────────────────────────────
    sort_cols = ["subj_f1_macro_mean","subj_bal_acc_mean","subj_auc_mean",
                 "f1_macro_mean","bal_acc_mean","auc_mean"]

    if early_all:
        df_early = (pd.concat(early_all, ignore_index=True)
                      .sort_values(sort_cols, ascending=[False]*len(sort_cols)))
        df_early.to_csv(OUT_DIR_EARLY / "early_all_ranked.csv",
                        index=False, float_format=CSV_FLOAT_FMT)

        # Ranking de familias de modelos
        df_early["model_family"] = df_early["model"].str.split("_g").str[0]
        ranking_familias = (df_early.groupby("model_family")[sort_cols[:3]]
                            .mean(numeric_only=True)
                            .sort_values(sort_cols[0], ascending=False))
        ranking_familias.to_csv(OUT_DIR_EARLY / "early_ranking_familias_modelo.csv",
                                float_format=CSV_FLOAT_FMT)

        display_cols = ["scope","pipeline_variant","tag","feature_set","model",
                        "subj_f1_macro_mean","subj_bal_acc_mean","subj_auc_mean",
                        "f1_macro_mean","bal_acc_mean","auc_mean","n_features","n_subjects"]
        print("\n── TOP 20 EARLY FUSION (ordenados por subj_f1_macro) ──")
        print(df_early[[c for c in display_cols if c in df_early.columns]]
              .head(20).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

        print("\n── RANKING DE FAMILIAS DE MODELOS ──")
        print(ranking_familias.to_string(float_format=lambda x: f"{x:.4f}"))

    if late_all:
        df_late = pd.concat(late_all, ignore_index=True)
        df_late.to_csv(OUT_DIR_LATE / "late_all_raw.csv",
                       index=False, float_format=CSV_FLOAT_FMT)
        valid = df_late[df_late["n_folds"] >= MIN_VALID_FOLDS_FOR_MAIN_RANKING].copy()
        if not valid.empty:
            valid = valid.sort_values(sort_cols, ascending=[False]*len(sort_cols))
            valid.to_csv(OUT_DIR_LATE / "late_all_ranked_valid.csv",
                         index=False, float_format=CSV_FLOAT_FMT)

            display_cols = ["global_scope","pipeline_variant","tag","ensemble",
                            "base_h1_feature_set","base_h1_model",
                            "base_h2_feature_set","base_h2_model",
                            "weight_h1","n_folds",
                            "subj_f1_macro_mean","subj_bal_acc_mean","subj_auc_mean",
                            "f1_macro_mean","bal_acc_mean","auc_mean","threshold_mean"]
            print("\n── TOP 25 LATE FUSION (n_folds ≥ 4) ──")
            print(valid[[c for c in display_cols if c in valid.columns]]
                  .head(25).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

            # Ranking por tipo de ensemble
            ens_ranking = (df_late.groupby("ensemble")[sort_cols[:3]]
                           .mean(numeric_only=True)
                           .sort_values(sort_cols[0], ascending=False))
            ens_ranking.to_csv(OUT_DIR_LATE / "late_ranking_por_ensemble.csv",
                               float_format=CSV_FLOAT_FMT)
            print("\n── RANKING POR TIPO DE ENSEMBLE ──")
            print(ens_ranking.to_string(float_format=lambda x: f"{x:.4f}"))

        print("\nDistribución n_folds (late):")
        print(df_late["n_folds"].value_counts(dropna=False).sort_index().to_string())

    elapsed = time.time() - t0
    print(f"\n{'='*62}")
    print(f"DONE.  Tiempo total: {elapsed/60:.1f} min")
    print(f"{'='*62}")


main()

---
## SECCIÓN 9 — Calibración isotónica y Rank Averaging

### ¿Por qué calibrar?
El blend con `w_GSG=0.85` da el máximo puntual aunque global spectral geometry sea más débil en media.
Una causa probable: dominant-mode spatial distribution produce probabilidades muy centradas en 0.5 (buen AUC pero
probs "tímidas"), mientras que global spectral geometry empuja más hacia 0 y 1.
Calibrar con **IsotonicRegression** sobre los OOF alinea las escalas antes de combinar.

### ¿Por qué rank averaging?
En lugar de promediar probabilidades (que dependen de calibración),
se promedian los **rankings** de los sujetos: invariante a escala y más robusto.

Ensembles nuevos que produce esta sección:
| Ensemble | Descripción |
|---|---|
| `blend_isotonic_w{0.3,0.5,0.7,0.85}` | blend después de calibración isotónica |
| `rank_avg_equal` | promedio de rankings con peso igual |
| `rank_avg_h2_heavy` | rankings con más peso a dominant-mode spatial distribution (w_GSG=0.30) |
| `rank_avg_h1_heavy` | rankings con más peso a global spectral geometry (w_GSG=0.70) |
| `rank_avg_h1_dominant` | rankings con global spectral geometry dominante (w_GSG=0.85) |


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECCIÓN 9 — Calibración isotónica + Rank Averaging
# ══════════════════════════════════════════════════════════════════════════
from sklearn.isotonic import IsotonicRegression
from scipy.stats import rankdata

OUT_DIR_COMP = BASE / "FUSION_ML_COMPLETO"
OUT_DIR_COMP.mkdir(parents=True, exist_ok=True)

BLEND_WEIGHTS_ISOTONIC = [0.30, 0.50, 0.70, 0.85]


def _calibrate_isotonic_oof(oof_tr, y_tr, scores_te):
    """Calibra `scores_te` usando IsotonicRegression ajustada sobre los OOF de train."""
    ir = IsotonicRegression(out_of_bounds="clip")
    ir.fit(oof_tr.reshape(-1, 1), y_tr)
    return ir.predict(scores_te.reshape(-1, 1))


def _rank_norm(scores):
    """Convierte scores a rango uniforme en [0,1]. Invariante a calibración."""
    n = len(scores)
    if n <= 1:
        return scores
    return (rankdata(scores) - 1.0) / max(n - 1, 1)


def _make_calib_rank_summary(fold_rows):
    """Agrega fold_rows al mismo formato de summary que _late_evaluate_block."""
    if not fold_rows:
        return pd.DataFrame()
    dff = pd.DataFrame(fold_rows)
    group_cols = ["tag", "scope", "pipeline_variant", "ensemble", "weight_h1"]
    summary = []
    for keys, g in dff.groupby(group_cols, dropna=False):
        summary.append({
            "tag": keys[0], "scope": keys[1], "pipeline_variant": keys[2],
            "ensemble": keys[3], "weight_h1": keys[4],
            "n_folds":            int(len(g)),
            "bal_acc_mean":       float(np.nanmean(g["bal_acc"])),
            "bal_acc_std":        float(np.nanstd(g["bal_acc"], ddof=1)) if len(g) > 1 else np.nan,
            "f1_macro_mean":      float(np.nanmean(g["f1_macro"])),
            "f1_macro_std":       float(np.nanstd(g["f1_macro"], ddof=1)) if len(g) > 1 else np.nan,
            "auc_mean":           float(np.nanmean(g["auc"])),
            "auc_std":            float(np.nanstd(g["auc"], ddof=1)) if len(g) > 1 else np.nan,
            "subj_bal_acc_mean":  float(np.nanmean(g["subj_bal_acc"])),
            "subj_bal_acc_std":   float(np.nanstd(g["subj_bal_acc"], ddof=1)) if len(g) > 1 else np.nan,
            "subj_f1_macro_mean": float(np.nanmean(g["subj_f1_macro"])),
            "subj_f1_macro_std":  float(np.nanstd(g["subj_f1_macro"], ddof=1)) if len(g) > 1 else np.nan,
            "subj_auc_mean":      float(np.nanmean(g["subj_auc"])),
            "subj_auc_std":       float(np.nanstd(g["subj_auc"], ddof=1)) if len(g) > 1 else np.nan,
            "threshold_mean":     float(np.nanmean(g["threshold"])),
            "threshold_std":      float(np.nanstd(g["threshold"], ddof=1)) if len(g) > 1 else np.nan,
            "base_h1_feature_set": most_frequent_nonempty(g["base_h1_feature_set"]),
            "base_h1_model":       most_frequent_nonempty(g["base_h1_model"]),
            "base_h2_feature_set": most_frequent_nonempty(g["base_h2_feature_set"]),
            "base_h2_model":       most_frequent_nonempty(g["base_h2_model"]),
        })
    return pd.DataFrame(summary)


def _calib_rank_evaluate_block(df_h1_block, df_h2_block, tag, scope_label, pv):
    """
    Extiende _late_evaluate_block con:
    1. blend_isotonic_w{w}: blend después de calibración isotónica por OOF.
    2. rank_avg_*: promedio de rankings con distintos pesos.
    Reutiliza exactamente la misma lógica de selección de base models.
    """
    if "id" not in df_h1_block.columns or "y" not in df_h1_block.columns:
        return pd.DataFrame()

    ids = df_h1_block["id"].astype(str).values
    y   = df_h1_block["y"].astype(int).values

    tmp = pd.DataFrame({"g": ids, "y": y}).drop_duplicates("g")
    if tmp["y"].nunique() < 2:
        return pd.DataFrame()
    if (int((tmp["y"] == 0).sum()) < MIN_GROUPS_PER_CLASS or
            int((tmp["y"] == 1).sum()) < MIN_GROUPS_PER_CLASS):
        return pd.DataFrame()

    cv = make_group_cv(y, ids, n_splits_desired=N_SPLITS_DESIRED)
    if cv is None:
        return pd.DataFrame()

    is_delta     = ("cond" not in df_h1_block.columns) and ("win_sec" in df_h1_block.columns)
    h1_sets      = build_late_h1_delta(df_h1_block) if is_delta else build_late_h1_static(df_h1_block)
    h2_sets      = build_late_h2_delta(df_h2_block) if is_delta else build_late_h2_static(df_h2_block)
    meta_cols    = {"id", "y", "cond", "win_sec", "tercile"}
    h1_feat_cols = [c for c in df_h1_block.columns if c not in meta_cols and c.startswith("h1__")]
    h2_feat_cols = [c for c in df_h2_block.columns if c not in meta_cols and c.startswith("h2__")]
    Xh1_all = df_h1_block[h1_feat_cols].copy()
    Xh2_all = df_h2_block[h2_feat_cols].copy()

    fold_rows = []

    for fold_idx, (tr, te) in enumerate(cv.split(np.zeros(len(y)), y, ids), 1):
        ids_tr, ids_te = ids[tr], ids[te]
        ytr, yte       = y[tr], y[te]

        best_h1 = select_best_base_model(Xh1_all.iloc[tr].copy(), ytr, ids_tr, h1_sets)
        if best_h1 is None: continue
        best_h2 = select_best_base_model(Xh2_all.iloc[tr].copy(), ytr, ids_tr, h2_sets)
        if best_h2 is None: continue

        Xh1_tr = build_X_numeric(Xh1_all.iloc[tr], best_h1["features_used"])
        Xh1_te = build_X_numeric(Xh1_all.iloc[te], best_h1["features_used"])
        _, s_h1_te = fit_predict_scores(clone(best_h1["model"]), Xh1_tr, ytr, Xh1_te)
        if s_h1_te is None: continue
        s_h1_te = to_prob_like(s_h1_te)
        oof_h1  = get_oof_scores(clone(best_h1["model"]), Xh1_tr, ytr, ids_tr)
        if oof_h1 is None: continue

        Xh2_tr = build_X_numeric(Xh2_all.iloc[tr], best_h2["features_used"])
        Xh2_te = build_X_numeric(Xh2_all.iloc[te], best_h2["features_used"])
        _, s_h2_te = fit_predict_scores(clone(best_h2["model"]), Xh2_tr, ytr, Xh2_te)
        if s_h2_te is None: continue
        s_h2_te = to_prob_like(s_h2_te)
        oof_h2  = get_oof_scores(clone(best_h2["model"]), Xh2_tr, ytr, ids_tr)
        if oof_h2 is None: continue

        base_info = dict(
            tag=tag, scope=scope_label, pipeline_variant=pv, fold=fold_idx,
            base_h1_feature_set=best_h1["feature_set"], base_h1_model=best_h1["model_name"],
            base_h2_feature_set=best_h2["feature_set"], base_h2_model=best_h2["model_name"],
        )

        # ── 1. Blend con calibración isotónica ────────────────────────────────
        # Se calibra cada modelo con su propio OOF sobre train, luego se blend en test.
        try:
            s_h1_cal = _calibrate_isotonic_oof(oof_h1, ytr, s_h1_te)
            s_h2_cal = _calibrate_isotonic_oof(oof_h2, ytr, s_h2_te)
            # Threshold se busca en los OOF calibrados sobre train
            oof_h1_cal = _calibrate_isotonic_oof(oof_h1, ytr, oof_h1)
            oof_h2_cal = _calibrate_isotonic_oof(oof_h2, ytr, oof_h2)
            for w in BLEND_WEIGHTS_ISOTONIC:
                blend_cal_tr = w * oof_h1_cal + (1.0 - w) * oof_h2_cal
                thr_cal, _   = choose_best_threshold(ytr, blend_cal_tr)
                blend_cal_te = w * s_h1_cal + (1.0 - w) * s_h2_cal
                fold_rows.append({
                    **base_info,
                    "ensemble":  f"blend_isotonic_w{w}",
                    "weight_h1": w,
                    "threshold": thr_cal,
                    **evaluate_scores(yte, ids_te, blend_cal_te, thr_cal),
                })
        except Exception:
            pass

        # ── 2. Rank Averaging ─────────────────────────────────────────────────
        # Ranking sobre test normalizado a [0,1]; threshold buscado en OOF rankeados.
        try:
            r_h1_te = _rank_norm(s_h1_te)
            r_h2_te = _rank_norm(s_h2_te)
            r_h1_tr = _rank_norm(oof_h1)
            r_h2_tr = _rank_norm(oof_h2)
            for w, name in [
                (0.50, "rank_avg_equal"),
                (0.30, "rank_avg_h2_heavy"),
                (0.70, "rank_avg_h1_heavy"),
                (0.85, "rank_avg_h1_dominant"),
            ]:
                blend_r_tr = w * r_h1_tr + (1.0 - w) * r_h2_tr
                thr_r, _   = choose_best_threshold(ytr, blend_r_tr)
                blend_r_te = w * r_h1_te + (1.0 - w) * r_h2_te
                fold_rows.append({
                    **base_info,
                    "ensemble":  name,
                    "weight_h1": w,
                    "threshold": thr_r,
                    **evaluate_scores(yte, ids_te, blend_r_te, thr_r),
                })
        except Exception:
            pass

    return _make_calib_rank_summary(fold_rows)


def run_calib_rank_bywin(df_h1, df_h2, pv):
    return _late_run(
        prep_h1_bywin(df_h1), prep_h2_bywin(df_h2, pv),
        merge_keys=["id", "y", "cond", "win_sec", "tercile"],
        groupby_cols=["cond", "win_sec", "tercile"],
        tag_prefix=f"calib_rank_bywin__{pv}", scope_label="bywin", pv=pv)
    # Note: _late_run calls _late_evaluate_block; we override via the wrapper below


def _run_calib_rank(h1, h2, merge_keys, groupby_cols, tag_prefix, scope_label, pv):
    """Versión de _late_run que usa _calib_rank_evaluate_block."""
    dfm = h1.merge(h2, on=merge_keys, how="inner")
    if dfm.empty: return pd.DataFrame()
    h1_cols = [c for c in h1.columns if c not in merge_keys]
    h2_cols = [c for c in h2.columns if c not in merge_keys]
    results = []
    for keys, d in dfm.groupby(groupby_cols, dropna=False):
        if len(d) < MIN_ROWS_PER_BLOCK or d["id"].nunique() < MIN_SUBJECTS_PER_BLOCK:
            continue
        tag = tag_prefix + "__" + "__".join(
            str(k) for k in (keys if isinstance(keys, tuple) else (keys,)))
        out = _calib_rank_evaluate_block(
            d[merge_keys + h1_cols].copy(),
            d[merge_keys + h2_cols].copy(),
            tag=tag, scope_label=scope_label, pv=pv)
        if not out.empty: results.append(out)
    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()


def run_calib_rank_bywin(df_h1, df_h2, pv):
    return _run_calib_rank(
        prep_h1_bywin(df_h1), prep_h2_bywin(df_h2, pv),
        merge_keys=["id", "y", "cond", "win_sec", "tercile"],
        groupby_cols=["cond", "win_sec", "tercile"],
        tag_prefix=f"calib_rank_bywin__{pv}", scope_label="bywin", pv=pv)


def run_calib_rank_pooled(df_h1, df_h2, pv):
    return _run_calib_rank(
        prep_h1_pooled(df_h1), prep_h2_pooled(df_h2, pv),
        merge_keys=["id", "y", "cond", "tercile"],
        groupby_cols=["cond", "tercile"],
        tag_prefix=f"calib_rank_pooled__{pv}", scope_label="pooled", pv=pv)


def run_calib_rank_delta(df_h1, df_h2, pv):
    return _run_calib_rank(
        prep_h1_delta(df_h1), prep_h2_delta(df_h2, pv),
        merge_keys=["id", "y", "win_sec", "tercile"],
        groupby_cols=["win_sec", "tercile"],
        tag_prefix=f"calib_rank_delta__{pv}", scope_label="delta", pv=pv)


print("Sección 9 OK — calibración isotónica + rank averaging")
print(f"  Pesos blend isotónico: {BLEND_WEIGHTS_ISOTONIC}")
print("  Rank ensembles: rank_avg_equal, rank_avg_h2_heavy, rank_avg_h1_heavy, rank_avg_h1_dominant")


---
## SECCIÓN 10 — Selección dinámica de experto (DCS-kNN + Router LR)

### ¿Por qué?
global spectral geometry y dominant-mode spatial distribution no rinden igual en todos los sujetos. global spectral geometry gana en algunos bloques (BASAL),
dominant-mode spatial distribution en otros (PVT). En lugar de combinarlos siempre con el mismo peso,
elegimos **cuál usar por sujeto**.

### DCS-kNN (Dynamic Classifier Selection)
Para cada sujeto de test, buscamos sus `k` vecinos más cercanos en el espacio
fusionado global spectral geometry+dominant-mode spatial distribution (train). Calculamos qué modelo acertó más en esos vecinos y
usamos ese modelo para el sujeto de test.

### Router LR
Entrenamos un clasificador LR cuya tarea es predecir si para un sujeto
conviene usar global spectral geometry o dominant-mode spatial distribution. Versión `hard` = elige uno; versión `soft` = interpola
según la probabilidad del router.

| Ensemble | Descripción |
|---|---|
| `dcs_knn_k3/5/7/11` | DCS con k vecinos |
| `router_LR_hard` | elige global spectral geometry o dominant-mode spatial distribution según LR router |
| `router_LR_soft` | interpola global spectral geometry/dominant-mode spatial distribution según prob del router |


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECCIÓN 10 — Selección dinámica de experto (DCS-kNN + Router LR)
# ══════════════════════════════════════════════════════════════════════════
from sklearn.neighbors import KNeighborsClassifier

KNN_K_VALUES = [3, 5, 7, 11]


def _dcs_evaluate_block(df_h1_block, df_h2_block, tag, scope_label, pv):
    """
    Dynamic Classifier Selection basado en kNN.
    Para cada sujeto de test:
      1. Busca sus k vecinos en el espacio fusionado combined eigenmode representations escalado (train).
      2. Calcula accuracy de global spectral geometry y dominant-mode spatial distribution en esos k vecinos (usando OOF de train).
      3. Elige el modelo con mejor accuracy local.
    Router LR: LR entrenado para predecir si conviene global spectral geometry o dominant-mode spatial distribution por sujeto.
    """
    if "id" not in df_h1_block.columns or "y" not in df_h1_block.columns:
        return pd.DataFrame()

    ids = df_h1_block["id"].astype(str).values
    y   = df_h1_block["y"].astype(int).values

    tmp = pd.DataFrame({"g": ids, "y": y}).drop_duplicates("g")
    if tmp["y"].nunique() < 2:
        return pd.DataFrame()
    if (int((tmp["y"] == 0).sum()) < MIN_GROUPS_PER_CLASS or
            int((tmp["y"] == 1).sum()) < MIN_GROUPS_PER_CLASS):
        return pd.DataFrame()

    cv = make_group_cv(y, ids, n_splits_desired=N_SPLITS_DESIRED)
    if cv is None:
        return pd.DataFrame()

    is_delta     = ("cond" not in df_h1_block.columns) and ("win_sec" in df_h1_block.columns)
    h1_sets      = build_late_h1_delta(df_h1_block) if is_delta else build_late_h1_static(df_h1_block)
    h2_sets      = build_late_h2_delta(df_h2_block) if is_delta else build_late_h2_static(df_h2_block)
    meta_cols    = {"id", "y", "cond", "win_sec", "tercile"}
    h1_feat_cols = [c for c in df_h1_block.columns if c not in meta_cols and c.startswith("h1__")]
    h2_feat_cols = [c for c in df_h2_block.columns if c not in meta_cols and c.startswith("h2__")]
    Xh1_all = df_h1_block[h1_feat_cols].copy()
    Xh2_all = df_h2_block[h2_feat_cols].copy()

    fold_rows = []

    for fold_idx, (tr, te) in enumerate(cv.split(np.zeros(len(y)), y, ids), 1):
        ids_tr, ids_te = ids[tr], ids[te]
        ytr, yte       = y[tr], y[te]

        best_h1 = select_best_base_model(Xh1_all.iloc[tr].copy(), ytr, ids_tr, h1_sets)
        if best_h1 is None: continue
        best_h2 = select_best_base_model(Xh2_all.iloc[tr].copy(), ytr, ids_tr, h2_sets)
        if best_h2 is None: continue

        Xh1_tr = build_X_numeric(Xh1_all.iloc[tr], best_h1["features_used"])
        Xh1_te = build_X_numeric(Xh1_all.iloc[te], best_h1["features_used"])
        _, s_h1_te = fit_predict_scores(clone(best_h1["model"]), Xh1_tr, ytr, Xh1_te)
        if s_h1_te is None: continue
        s_h1_te = to_prob_like(s_h1_te)
        oof_h1  = get_oof_scores(clone(best_h1["model"]), Xh1_tr, ytr, ids_tr)
        if oof_h1 is None: continue

        Xh2_tr = build_X_numeric(Xh2_all.iloc[tr], best_h2["features_used"])
        Xh2_te = build_X_numeric(Xh2_all.iloc[te], best_h2["features_used"])
        _, s_h2_te = fit_predict_scores(clone(best_h2["model"]), Xh2_tr, ytr, Xh2_te)
        if s_h2_te is None: continue
        s_h2_te = to_prob_like(s_h2_te)
        oof_h2  = get_oof_scores(clone(best_h2["model"]), Xh2_tr, ytr, ids_tr)
        if oof_h2 is None: continue

        base_info = dict(
            tag=tag, scope=scope_label, pipeline_variant=pv, fold=fold_idx,
            base_h1_feature_set=best_h1["feature_set"], base_h1_model=best_h1["model_name"],
            base_h2_feature_set=best_h2["feature_set"], base_h2_model=best_h2["model_name"],
        )

        # ── Espacio fusionado para kNN: imputa + escala global spectral geometry || dominant-mode spatial distribution ──────────────
        try:
            from sklearn.impute import SimpleImputer as _SI
            from sklearn.preprocessing import StandardScaler as _SS
            imp1, ss1 = _SI(strategy="median"), _SS()
            imp2, ss2 = _SI(strategy="median"), _SS()
            Xfus_tr = np.hstack([
                ss1.fit_transform(imp1.fit_transform(Xh1_tr.values)),
                ss2.fit_transform(imp2.fit_transform(Xh2_tr.values)),
            ])
            Xfus_te = np.hstack([
                ss1.transform(imp1.transform(Xh1_te.values)),
                ss2.transform(imp2.transform(Xh2_te.values)),
            ])
        except Exception:
            continue

        # Aciertos de global spectral geometry y dominant-mode spatial distribution en OOF (para el selector kNN)
        correct_h1 = ((oof_h1 >= 0.5).astype(int) == ytr).astype(int)
        correct_h2 = ((oof_h2 >= 0.5).astype(int) == ytr).astype(int)

        # Threshold proxy: blend 50/50 de OOF
        thr_proxy, _ = choose_best_threshold(ytr, 0.5 * oof_h1 + 0.5 * oof_h2)

        # ── DCS-kNN ───────────────────────────────────────────────────────────
        for k in KNN_K_VALUES:
            if k >= len(Xfus_tr):
                continue
            try:
                knn = KNeighborsClassifier(n_neighbors=k, metric="euclidean")
                knn.fit(Xfus_tr, np.zeros(len(Xfus_tr)))  # solo para vecinos
                _, neighb_idx = knn.kneighbors(Xfus_te)

                s_dcs = np.zeros(len(yte))
                for i_te, nbrs in enumerate(neighb_idx):
                    acc_h1_local = correct_h1[nbrs].mean()
                    acc_h2_local = correct_h2[nbrs].mean()
                    # Empate → dominant-mode spatial distribution (más fuerte en media global)
                    s_dcs[i_te] = s_h1_te[i_te] if acc_h1_local > acc_h2_local else s_h2_te[i_te]

                fold_rows.append({
                    **base_info,
                    "ensemble":  f"dcs_knn_k{k}",
                    "weight_h1": np.nan,
                    "threshold": thr_proxy,
                    **evaluate_scores(yte, ids_te, s_dcs, thr_proxy),
                })
            except Exception:
                pass

        # ── Router LR ─────────────────────────────────────────────────────────
        # Etiqueta del router: 1 si global spectral geometry acierta y dominant-mode spatial distribution falla, 0 en cualquier otro caso
        try:
            router_y = ((correct_h1 == 1) & (correct_h2 == 0)).astype(int)
            # Necesitamos al menos 3 ejemplos de cada clase para el router
            if router_y.sum() >= 3 and (len(router_y) - router_y.sum()) >= 3:
                router_clf = LogisticRegression(
                    solver="liblinear", C=1.0, class_weight="balanced",
                    random_state=SEED, max_iter=8000)
                router_clf.fit(Xfus_tr, router_y)
                use_h1_prob = router_clf.predict_proba(Xfus_te)[:, 1]

                # Hard: elige global spectral geometry si el router dice p > 0.5
                s_hard = np.where(use_h1_prob > 0.5, s_h1_te, s_h2_te)
                fold_rows.append({
                    **base_info, "ensemble": "router_LR_hard", "weight_h1": np.nan,
                    "threshold": thr_proxy,
                    **evaluate_scores(yte, ids_te, s_hard, thr_proxy),
                })

                # Soft: interpola según la probabilidad del router
                s_soft = use_h1_prob * s_h1_te + (1.0 - use_h1_prob) * s_h2_te
                fold_rows.append({
                    **base_info, "ensemble": "router_LR_soft", "weight_h1": np.nan,
                    "threshold": thr_proxy,
                    **evaluate_scores(yte, ids_te, s_soft, thr_proxy),
                })
        except Exception:
            pass

    return _make_calib_rank_summary(fold_rows)


def _run_dcs(h1, h2, merge_keys, groupby_cols, tag_prefix, scope_label, pv):
    """Versión de _late_run que usa _dcs_evaluate_block."""
    dfm = h1.merge(h2, on=merge_keys, how="inner")
    if dfm.empty: return pd.DataFrame()
    h1_cols = [c for c in h1.columns if c not in merge_keys]
    h2_cols = [c for c in h2.columns if c not in merge_keys]
    results = []
    for keys, d in dfm.groupby(groupby_cols, dropna=False):
        if len(d) < MIN_ROWS_PER_BLOCK or d["id"].nunique() < MIN_SUBJECTS_PER_BLOCK:
            continue
        tag = tag_prefix + "__" + "__".join(
            str(k) for k in (keys if isinstance(keys, tuple) else (keys,)))
        out = _dcs_evaluate_block(
            d[merge_keys + h1_cols].copy(),
            d[merge_keys + h2_cols].copy(),
            tag=tag, scope_label=scope_label, pv=pv)
        if not out.empty: results.append(out)
    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()


def run_dcs_bywin(df_h1, df_h2, pv):
    return _run_dcs(
        prep_h1_bywin(df_h1), prep_h2_bywin(df_h2, pv),
        merge_keys=["id", "y", "cond", "win_sec", "tercile"],
        groupby_cols=["cond", "win_sec", "tercile"],
        tag_prefix=f"dcs_bywin__{pv}", scope_label="bywin", pv=pv)


def run_dcs_pooled(df_h1, df_h2, pv):
    return _run_dcs(
        prep_h1_pooled(df_h1), prep_h2_pooled(df_h2, pv),
        merge_keys=["id", "y", "cond", "tercile"],
        groupby_cols=["cond", "tercile"],
        tag_prefix=f"dcs_pooled__{pv}", scope_label="pooled", pv=pv)


def run_dcs_delta(df_h1, df_h2, pv):
    return _run_dcs(
        prep_h1_delta(df_h1), prep_h2_delta(df_h2, pv),
        merge_keys=["id", "y", "win_sec", "tercile"],
        groupby_cols=["win_sec", "tercile"],
        tag_prefix=f"dcs_delta__{pv}", scope_label="delta", pv=pv)


print("Sección 10 OK — DCS-kNN + Router LR")
print(f"  k values: {KNN_K_VALUES}")
print("  Ensembles: dcs_knn_k3/5/7/11, router_LR_hard, router_LR_soft")


---
## SECCIÓN 11 — Early Fusion con features de interacción global spectral geometry×dominant-mode spatial distribution

### ¿Por qué?
En Early Fusion se concatena [global spectral geometry | dominant-mode spatial distribution] y el clasificador descubre interacciones
solo si tiene suficiente capacidad y datos. Podemos darle una ventaja creando
explícitamente features del tipo `interaction terms between global spectral geometry and dominant-mode spatial distribution` para los pares más relevantes
según lo que sabemos del análisis estadístico:

| Feature global spectral geometry | Feature dominant-mode spatial distribution | Por qué tiene sentido |
|---|---|---|
| `osc__rad_std` | `s_core` | dispersión oscilatoria × dominancia temporal-parietal |
| `osc__rad_p95` | `dom_gap_core_vs_best_other` | cola alta del radio × ventaja del core |
| `osc_frac` | `s_temporal` | fracción oscilatoria × participación temporal |
| `osc__imag_abs_mean` | `core_to_rest` | oscilación media × ratio core/resto |
| `all__dist_p95` | `entropy_norm` | dispersión global × entropía de distribución |

Feature sets que produce esta sección:
| Feature set | Contenido |
|---|---|
| `h1_statcompact__h2_dist__interactions` | global spectral geometry compacto + dominant-mode spatial distribution distribución + productos cruzados |
| `h1_core__h2_priority__interactions` | global spectral geometry mínimo + dominant-mode spatial distribution prioritarias + productos cruzados |
| `interactions_only` | solo los 10 productos cruzados (¿cuánto aportan solos?) |


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECCIÓN 11 — Early Fusion con features de interacción global spectral geometry×dominant-mode spatial distribution
# ══════════════════════════════════════════════════════════════════════════

# Pares de interacción: (nombre_H1_sin_prefijo, nombre_H2_sin_prefijo)
# Se calculan como productos cruzados: col_h1 * col_h2
INTERACTION_PAIRS = [
    ("osc__rad_std",            "s_core"),
    ("osc__rad_std",            "core_to_rest"),
    ("osc__rad_p95",            "dom_gap_core_vs_best_other"),
    ("osc__rad_p95",            "s_core"),
    ("osc_frac",                "s_temporal"),
    ("osc_frac",                "s_central"),
    ("osc__imag_abs_mean",      "dom_gap_core_vs_best_other"),
    ("osc__imag_abs_mean",      "core_to_rest"),
    ("all__dist_p95",           "entropy_norm"),
    ("all__dist_prop_gt_0.02",  "gini"),
]


def _add_interaction_features(df_merged, pairs):
    """
    Añade columnas de producto interaction terms between global spectral geometry and dominant-mode spatial distribution al df ya mergeado con prefijos h1__ y h2__.
    Devuelve (df_extendido, lista_nombres_nuevas_columnas).
    Solo crea el producto si ambas columnas existen y tienen varianza > MIN_VAR_PER_FEATURE.
    """
    df = df_merged.copy()
    new_cols = []
    for c1_base, c2_base in pairs:
        c1 = f"h1__{c1_base}"
        c2 = f"h2__{c2_base}"
        if c1 not in df.columns or c2 not in df.columns:
            continue
        x1 = pd.to_numeric(df[c1], errors="coerce")
        x2 = pd.to_numeric(df[c2], errors="coerce")
        if x1.var() < MIN_VAR_PER_FEATURE or x2.var() < MIN_VAR_PER_FEATURE:
            continue
        # Nombre del producto: ix__ para distinguirlos de las features base
        col_name = f"ix__{c1_base}__X__{c2_base}"
        df[col_name] = x1 * x2
        new_cols.append(col_name)
    return df, new_cols


def _build_interaction_fusion_sets(df_merged, ix_cols):
    """
    Construye los feature sets de Early Fusion con interacciones.
    Usa los mismos nombres de sets que el Early Fusion original pero + ix_cols.
    """
    sets = {}

    # global spectral geometry compacto + dominant-mode spatial distribution distribución + interacciones
    base1 = available_features(df_merged,
        [f"h1__{x}" for x in GSG_STAT_COMPACT] +
        [f"h2__{x}" for x in DMSD_DISTRIBUTION])
    if len(base1) >= MIN_FEATURES_AFTER_CLEAN:
        sets["h1_statcompact__h2_dist__interactions"] = dedupe_keep_order(base1 + ix_cols)

    # global spectral geometry core mínimo + dominant-mode spatial distribution priority_stat + interacciones
    base2 = available_features(df_merged,
        [f"h1__{x}" for x in GSG_CORE_ONLY] +
        [f"h2__{x}" for x in DMSD_PRIORITY_STAT])
    if len(base2) >= MIN_FEATURES_AFTER_CLEAN:
        sets["h1_core__h2_priority__interactions"] = dedupe_keep_order(base2 + ix_cols)

    # Solo las interacciones
    if len(ix_cols) >= MIN_FEATURES_AFTER_CLEAN:
        sets["interactions_only"] = ix_cols

    return sets


def _run_early_interaction(h1, h2, merge_keys, groupby_cols, tag_prefix, scope_label, pv):
    """
    Early Fusion con interacciones. Igual que _early_run pero:
    1. Calcula los productos global spectral geometry×dominant-mode spatial distribution sobre el df mergeado completo.
    2. Usa _build_interaction_fusion_sets para construir los feature sets.
    3. Llama a _early_evaluate_block (sin modificar) sobre cada feature set.
    """
    dfm = h1.merge(h2, on=merge_keys, how="inner")
    if dfm.empty: return pd.DataFrame()

    results = []
    for keys, d in dfm.groupby(groupby_cols, dropna=False):
        if len(d) < MIN_ROWS_PER_BLOCK or d["id"].nunique() < MIN_SUBJECTS_PER_BLOCK:
            continue
        tag_base = tag_prefix + "__" + "__".join(
            str(k) for k in (keys if isinstance(keys, tuple) else (keys,)))

        # Calcular interacciones sobre este bloque
        d_ix, ix_cols = _add_interaction_features(d, INTERACTION_PAIRS)
        if not ix_cols:
            continue

        fusion_sets = _build_interaction_fusion_sets(d_ix, ix_cols)
        for fs_name, fs_cols in fusion_sets.items():
            fs_cols = dedupe_keep_order([c for c in fs_cols if c in d_ix.columns])
            if len(fs_cols) < MIN_FEATURES_AFTER_CLEAN:
                continue
            out = _early_evaluate_block(
                d_ix, fs_cols,
                tag=f"{tag_base}__{fs_name}",
                fs_name=fs_name)
            if not out.empty:
                results.append(out)

    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()


def run_early_interaction_bywin(df_h1, df_h2, pv):
    return _run_early_interaction(
        prep_h1_bywin(df_h1), prep_h2_bywin(df_h2, pv),
        merge_keys=["id", "y", "cond", "win_sec", "tercile"],
        groupby_cols=["cond", "win_sec", "tercile"],
        tag_prefix=f"early_ix_bywin__{pv}", scope_label="bywin", pv=pv)


def run_early_interaction_pooled(df_h1, df_h2, pv):
    return _run_early_interaction(
        prep_h1_pooled(df_h1), prep_h2_pooled(df_h2, pv),
        merge_keys=["id", "y", "cond", "tercile"],
        groupby_cols=["cond", "tercile"],
        tag_prefix=f"early_ix_pooled__{pv}", scope_label="pooled", pv=pv)


def run_early_interaction_delta(df_h1, df_h2, pv):
    return _run_early_interaction(
        prep_h1_delta(df_h1), prep_h2_delta(df_h2, pv),
        merge_keys=["id", "y", "win_sec", "tercile"],
        groupby_cols=["win_sec", "tercile"],
        tag_prefix=f"early_ix_delta__{pv}", scope_label="delta", pv=pv)


print("Sección 11 OK — features de interacción global spectral geometry×dominant-mode spatial distribution")
print(f"  Pares definidos: {len(INTERACTION_PAIRS)}")
print("  Feature sets: h1_statcompact__h2_dist__interactions, "
      "h1_core__h2_priority__interactions, interactions_only")


---
## SECCIÓN 12 — Main extendido → FUSION_ML_COMPLETO

Ejecuta las cinco familias de estrategias y guarda todo en `FUSION_ML_COMPLETO/`.

| Estrategia | Scopes | Función principal |
|---|---|---|
| Early Fusion original | bywin, pooled, delta | `run_early_bywin/pooled/delta` |
| Late Fusion original | bywin, pooled, delta | `run_late_bywin/pooled/delta` |
| Calibración + Rank Avg | bywin, pooled, delta | `run_calib_rank_bywin/pooled/delta` |
| DCS-kNN + Router | bywin, pooled, delta | `run_dcs_bywin/pooled/delta` |
| Early Fusion interacciones | bywin, pooled, delta | `run_early_interaction_bywin/pooled/delta` |

CSV de salida en `FUSION_ML_COMPLETO/`:
- `early_*.csv`, `late_*.csv`, `calib_rank_*.csv`, `dcs_*.csv`, `early_ix_*.csv` — por scope/pipeline
- `early_all_ranked.csv`, `late_all_ranked_valid.csv` — resúmenes originales
- `calib_rank_all_ranked.csv`, `dcs_all_ranked.csv`, `early_ix_all_ranked.csv` — nuevos
- `ALL_STRATEGIES_RANKED.csv` — ranking global de todas las estrategias

> **Ejecutar las secciones 1–11 antes de llamar a `main_completo()`.**


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SECCIÓN 12 — Main extendido → FUSION_ML_COMPLETO
# ══════════════════════════════════════════════════════════════════════════

def _save_ranked(df, path, sort_cols):
    """Guarda un df ordenado por sort_cols descendente."""
    if df is None or df.empty: return
    (df.sort_values(sort_cols, ascending=[False] * len(sort_cols))
       .to_csv(path, index=False, float_format=CSV_FLOAT_FMT))


def _top_display(df, label, sort_cols, n=15):
    """Imprime el top-n de un df."""
    if df is None or df.empty:
        print(f"  {label}: sin resultados")
        return
    top = df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).head(n)
    disp_cols = ["pipeline_variant", "scope", "ensemble", "feature_set",
                 "subj_f1_macro_mean", "subj_bal_acc_mean", "subj_auc_mean"]
    cols = [c for c in disp_cols if c in top.columns]
    print(f"\n── TOP {n}: {label} ──")
    print(top[cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))


def main_completo():
    import time
    t0 = time.time()

    print("=" * 65)
    print("combined eigenmode representations  FUSION COMPLETO")
    print(f"  OUT → {OUT_DIR_COMP}")
    print(f"  Pipelines dominant-mode spatial distribution: {DMSD_PIPELINES}")
    print("=" * 65)

    DATA = load_all_data()
    sort_cols = ["subj_f1_macro_mean", "subj_bal_acc_mean", "subj_auc_mean",
                 "f1_macro_mean", "bal_acc_mean", "auc_mean"]

    # Colectores por familia
    early_all  = []
    late_all   = []
    calib_all  = []
    dcs_all    = []
    ix_all     = []

    for pv in DMSD_PIPELINES:
        print(f"\n{'─' * 60}")
        print(f"PIPELINE dominant-mode spatial distribution: {pv}")
        print(f"{'─' * 60}")

        # ── 1. EARLY FUSION ORIGINAL ─────────────────────────────────────────
        for scope, runner, h1k, h2k in [
            ("bywin",  run_early_bywin,  "h1_bywin",  "h2_bywin"),
            ("pooled", run_early_pooled, "h1_pooled", "h2_pooled"),
            ("delta",  run_early_delta,  "h1_delta",  "h2_delta"),
        ]:
            print(f"  [EARLY {scope}]...")
            r = runner(DATA[h1k], DATA[h2k], pv)
            if not r.empty:
                r.to_csv(OUT_DIR_COMP / f"early_{scope}__{pv}.csv",
                         index=False, float_format=CSV_FLOAT_FMT)
                save_early_summary(r, f"early_{scope}__{pv}")
                early_all.append(r.assign(scope=scope, pipeline_variant=pv))
                print(f"     → {len(r)} filas")

        # ── 2. LATE FUSION ORIGINAL ──────────────────────────────────────────
        for scope, runner, h1k, h2k in [
            ("bywin",  run_late_bywin,  "h1_bywin",  "h2_bywin"),
            ("pooled", run_late_pooled, "h1_pooled", "h2_pooled"),
            ("delta",  run_late_delta,  "h1_delta",  "h2_delta"),
        ]:
            print(f"  [LATE  {scope}]...")
            r = runner(DATA[h1k], DATA[h2k], pv)
            if not r.empty:
                r.to_csv(OUT_DIR_COMP / f"late_{scope}__{pv}.csv",
                         index=False, float_format=CSV_FLOAT_FMT)
                save_late_summary(r, f"late_{scope}__{pv}")
                late_all.append(r.assign(global_scope=scope))
                print(f"     → {len(r)} filas")

        # ── 3. CALIBRACIÓN ISOTÓNICA + RANK AVERAGING ────────────────────────
        for scope, runner, h1k, h2k in [
            ("bywin",  run_calib_rank_bywin,  "h1_bywin",  "h2_bywin"),
            ("pooled", run_calib_rank_pooled, "h1_pooled", "h2_pooled"),
            ("delta",  run_calib_rank_delta,  "h1_delta",  "h2_delta"),
        ]:
            print(f"  [CALIB {scope}]...")
            r = runner(DATA[h1k], DATA[h2k], pv)
            if not r.empty:
                r.to_csv(OUT_DIR_COMP / f"calib_rank_{scope}__{pv}.csv",
                         index=False, float_format=CSV_FLOAT_FMT)
                calib_all.append(r.assign(scope=scope, pipeline_variant=pv))
                print(f"     → {len(r)} filas")

        # ── 4. DCS-kNN + ROUTER ──────────────────────────────────────────────
        for scope, runner, h1k, h2k in [
            ("bywin",  run_dcs_bywin,  "h1_bywin",  "h2_bywin"),
            ("pooled", run_dcs_pooled, "h1_pooled", "h2_pooled"),
            ("delta",  run_dcs_delta,  "h1_delta",  "h2_delta"),
        ]:
            print(f"  [DCS   {scope}]...")
            r = runner(DATA[h1k], DATA[h2k], pv)
            if not r.empty:
                r.to_csv(OUT_DIR_COMP / f"dcs_{scope}__{pv}.csv",
                         index=False, float_format=CSV_FLOAT_FMT)
                dcs_all.append(r.assign(scope=scope, pipeline_variant=pv))
                print(f"     → {len(r)} filas")

        # ── 5. EARLY FUSION CON INTERACCIONES ────────────────────────────────
        for scope, runner, h1k, h2k in [
            ("bywin",  run_early_interaction_bywin,  "h1_bywin",  "h2_bywin"),
            ("pooled", run_early_interaction_pooled, "h1_pooled", "h2_pooled"),
            ("delta",  run_early_interaction_delta,  "h1_delta",  "h2_delta"),
        ]:
            print(f"  [IX    {scope}]...")
            r = runner(DATA[h1k], DATA[h2k], pv)
            if not r.empty:
                r.to_csv(OUT_DIR_COMP / f"early_ix_{scope}__{pv}.csv",
                         index=False, float_format=CSV_FLOAT_FMT)
                ix_all.append(r.assign(scope=scope, pipeline_variant=pv))
                print(f"     → {len(r)} filas")

    # ── Resúmenes por familia ──────────────────────────────────────────────────
    print(f"\n{'=' * 65}")
    print("RESÚMENES")
    print(f"{'=' * 65}")

    if early_all:
        df_early = pd.concat(early_all, ignore_index=True)
        _save_ranked(df_early, OUT_DIR_COMP / "early_all_ranked.csv", sort_cols)
        _top_display(df_early, "EARLY FUSION", sort_cols)

    if late_all:
        df_late = pd.concat(late_all, ignore_index=True)
        df_late.to_csv(OUT_DIR_COMP / "late_all_raw.csv",
                       index=False, float_format=CSV_FLOAT_FMT)
        valid = df_late[df_late["n_folds"] >= MIN_VALID_FOLDS_FOR_MAIN_RANKING]
        if not valid.empty:
            _save_ranked(valid, OUT_DIR_COMP / "late_all_ranked_valid.csv", sort_cols)
        ens_r = (df_late.groupby("ensemble")[sort_cols[:3]]
                 .mean(numeric_only=True)
                 .sort_values(sort_cols[0], ascending=False))
        ens_r.to_csv(OUT_DIR_COMP / "late_ranking_ensemble.csv",
                     float_format=CSV_FLOAT_FMT)
        _top_display(df_late, "LATE FUSION", sort_cols)
        print("\n── Ranking por ensemble (Late) ──")
        print(ens_r.to_string(float_format=lambda x: f"{x:.4f}"))

    if calib_all:
        df_calib = pd.concat(calib_all, ignore_index=True)
        _save_ranked(df_calib, OUT_DIR_COMP / "calib_rank_all_ranked.csv", sort_cols)
        cr = (df_calib.groupby("ensemble")[sort_cols[:3]]
              .mean(numeric_only=True)
              .sort_values(sort_cols[0], ascending=False))
        cr.to_csv(OUT_DIR_COMP / "calib_rank_ranking_ensemble.csv",
                  float_format=CSV_FLOAT_FMT)
        _top_display(df_calib, "CALIBRACIÓN + RANK AVG", sort_cols)
        print("\n── Ranking por ensemble (Calib+Rank) ──")
        print(cr.to_string(float_format=lambda x: f"{x:.4f}"))

    if dcs_all:
        df_dcs = pd.concat(dcs_all, ignore_index=True)
        _save_ranked(df_dcs, OUT_DIR_COMP / "dcs_all_ranked.csv", sort_cols)
        dr = (df_dcs.groupby("ensemble")[sort_cols[:3]]
              .mean(numeric_only=True)
              .sort_values(sort_cols[0], ascending=False))
        dr.to_csv(OUT_DIR_COMP / "dcs_ranking_ensemble.csv",
                  float_format=CSV_FLOAT_FMT)
        _top_display(df_dcs, "DCS-kNN + ROUTER", sort_cols)
        print("\n── Ranking por ensemble (DCS) ──")
        print(dr.to_string(float_format=lambda x: f"{x:.4f}"))

    if ix_all:
        df_ix = pd.concat(ix_all, ignore_index=True)
        _save_ranked(df_ix, OUT_DIR_COMP / "early_ix_all_ranked.csv", sort_cols)
        _top_display(df_ix, "EARLY FUSION CON INTERACCIONES", sort_cols)

    # ── Ranking global de todas las estrategias ────────────────────────────────
    all_dfs = []
    for df, label in [
        (pd.concat(early_all,  ignore_index=True) if early_all  else None, "early"),
        (pd.concat(late_all,   ignore_index=True) if late_all   else None, "late"),
        (pd.concat(calib_all,  ignore_index=True) if calib_all  else None, "calib_rank"),
        (pd.concat(dcs_all,    ignore_index=True) if dcs_all    else None, "dcs"),
        (pd.concat(ix_all,     ignore_index=True) if ix_all     else None, "early_ix"),
    ]:
        if df is not None and not df.empty:
            all_dfs.append(df.assign(strategy=label))

    if all_dfs:
        df_all = pd.concat(all_dfs, ignore_index=True)
        _save_ranked(df_all, OUT_DIR_COMP / "ALL_STRATEGIES_RANKED.csv", sort_cols)
        top_global = df_all.sort_values(sort_cols, ascending=[False] * len(sort_cols)).head(25)
        gc = ["strategy", "pipeline_variant", "scope", "ensemble", "feature_set",
              "subj_f1_macro_mean", "subj_bal_acc_mean", "subj_auc_mean"]
        print("\n" + "=" * 65)
        print("TOP 25 GLOBAL — todas las estrategias")
        print("=" * 65)
        print(top_global[[c for c in gc if c in top_global.columns]]
              .to_string(index=False, float_format=lambda x: f"{x:.4f}"))
        print(f"\nCSV global: {OUT_DIR_COMP / 'ALL_STRATEGIES_RANKED.csv'}")

    elapsed = time.time() - t0
    print(f"\n{'=' * 65}")
    print(f"DONE. Tiempo total: {elapsed / 60:.1f} min")
    print(f"Resultados en: {OUT_DIR_COMP}")
    print(f"{'=' * 65}")


main_completo()
